# NB_SETUP_INITIAL_DDL

Dựng một lần môi trường trống: DDL đầy đủ và seed cấu hình, chỉ thêm dòng chưa có. `p_apply` mặc định `false`: in danh sách DDL và số dòng sẽ thêm, rồi thoát, không ghi.

Điều kiện: `lh_vv_ctrl`, `lh_vv_silver`, `lh_vv_gold` đã có schema `dbo`, chưa có bảng cần tạo, và không có pipeline đang chạy. Notebook không xoá bảng.

Vai trò với notebook vá: `docs/context/02_NAMING_CONVENTION.md` §2.1.


In [ ]:
p_apply = "false"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
# BEGIN_INIT_TABLE_SPECS
PROPS = "'delta.autoOptimize.optimizeWrite' = 'true', 'delta.autoOptimize.autoCompact' = 'true'"
# Mỗi bảng: định nghĩa đầy đủ hiện hành. Cột thêm sau ở notebook vá ghi thẳng vào đây.
# cols: (tên, kiểu Spark SQL kèm NOT NULL nếu có, comment hoặc None)
TABLE_SPECS = [
    {"fq": 'lh_vv_ctrl.dbo.ctrl_mng_pipeline_config', "partition": None, "comment": None, "props": None,
     "cols": [
        ('id', 'BIGINT', None),
        ('pl_name', 'STRING', None),
        ('src_schema', 'STRING', None),
        ('src_tbl', 'STRING', None),
        ('trg_schema', 'STRING', None),
        ('trg_tbl', 'STRING', None),
        ('is_active', 'INT', None),
        ('priority', 'INT', None),
        ('load_mode', 'STRING', 'CDC (Debezium, partner) | DOC | DOC_ARRAY | LANG | LANG_ARRAY (snapshot, 3rd-party). NULL = CDC'),
        ('align_path', 'STRING', 'DOC_ARRAY / LANG_ARRAY: mảng tính từ gốc payload ghép theo vị trí, đọc bằng json_path _align.<trường>'),
        ('dedup_order', 'STRING', 'Snapshot: chọn 1 dòng khi 1 tài liệu có nhiều dòng cùng khoá, vd is_original DESC, created_at DESC'),
        ('nb_name', 'STRING', None),
     ]},
    {"fq": 'lh_vv_ctrl.dbo.ctrl_mng_watermark', "partition": None, "comment": None, "props": None,
     "cols": [
        ('watermark_id', 'STRING', None),
        ('src_schema', 'STRING', None),
        ('src_tbl', 'STRING', None),
        ('trg_schema', 'STRING', None),
        ('trg_tbl', 'STRING', None),
        ('watermark_column', 'STRING', None),
        ('watermark_value', 'TIMESTAMP', None),
        ('last_success_at', 'TIMESTAMP', None),
        ('status', 'STRING', None),
        ('flow_name', 'STRING', None),
        ('watermark_sequence', 'BIGINT', None),
        ('last_run_id', 'STRING', None),
        ('last_src_version', 'BIGINT', None),
        ('last_src_table_id', 'STRING', None),
        ('error_message', 'STRING', None),
        ('lock_exec_id', 'STRING', None),
        ('lock_at', 'TIMESTAMP', None),
        ('updated_at', 'TIMESTAMP', None),
     ]},
    {"fq": 'lh_vv_ctrl.dbo.ctrl_cfg_schema_registry', "partition": None, "comment": 'Mapping cột JSON sang bảng đích (CDC partner, snapshot 3rd-party) - 1 dòng / cột', "props": None,
     "cols": [
        ('src_schema', 'STRING', 'Schema bảng raw, vd lh_vv_bronze.dbo'),
        ('src_tbl', 'STRING', 'Bảng raw, vd partner_raw_data (khớp ctrl_mng_pipeline_config.src_tbl)'),
        ('src_object_schema', 'STRING', 'CDC: schema trong Debezium source.schema, vd public. Snapshot (3rd-party): NULL'),
        ('src_object', 'STRING', 'CDC: bảng trong Debezium source.table, vd products. Snapshot: khối trong payload, vd poi_address ($ = gốc payload)'),
        ('trg_schema', 'STRING', 'Schema bảng đích, vd lh_vv_silver.dbo'),
        ('trg_tbl', 'STRING', 'Bảng đích, khớp ctrl_mng_pipeline_config.trg_tbl'),
        ('trg_column', 'STRING', 'Tên cột trong bảng đích'),
        ('json_path', 'STRING', 'CDC: đường dẫn trong JSON before/after. Snapshot: đường dẫn cấu trúc; không NULL'),
        ('data_type', 'STRING', 'Kiểu Spark của cột đích: string, bigint, int, smallint, boolean, timestamp, decimal(19,4)...'),
        ('convert_arg', 'STRING', 'Snapshot rule argument as JSON. NULL for rules without arguments'),
        ('convert_rule', 'STRING', 'Cách chuyển giá trị gốc; snapshot thêm HASH_*, COALESCE, ARRAY_PICK, ARRAY_AT, ENUM_MAP, ARRAY_PROJECT, POS_1BASED, JSON_PICK'),
        ('is_key', 'BOOLEAN', 'Cột thuộc khoá MERGE'),
        ('key_order', 'INT', 'Thứ tự trong khoá ghép (1, 2...), NULL nếu không phải khoá'),
        ('is_toast', 'BOOLEAN', 'Cột có thể nhận giá trị TOAST thay thế của Debezium (chỉ CDC, snapshot luôn false)'),
        ('column_order', 'INT', 'Thứ tự cột trong bảng đích (1, 2...)'),
        ('is_active', 'BOOLEAN', 'Bật/tắt cột; bật/tắt cả bảng dùng ctrl_mng_pipeline_config.is_active'),
        ('description', 'STRING', 'Ghi chú'),
     ]},
    {"fq": 'lh_vv_ctrl.dbo.ctrl_log_run', "partition": None, "comment": 'Log mỗi lần chạy notebook extract CDC', "props": None,
     "cols": [
        ('exec_id', 'STRING', 'ID mỗi lần notebook chạy (uuid) - khoá'),
        ('run_id', 'STRING', 'RunId của pipeline (chạy tay: = exec_id)'),
        ('pl_name', 'STRING', 'Pipeline, khớp ctrl_mng_pipeline_config.pl_name'),
        ('nb_name', 'STRING', 'Notebook thực thi'),
        ('src_schema', 'STRING', 'Schema bảng nguồn raw, vd lh_vv_bronze.dbo'),
        ('src_tbl', 'STRING', 'Bảng nguồn raw, vd partner_raw_data'),
        ('watermark_id', 'STRING', 'Khớp ctrl_mng_watermark.watermark_id'),
        ('run_mode', 'STRING', 'INCREMENTAL | FULL_RELOAD | DRY_RUN'),
        ('table_filter', 'STRING', 'Danh sách trg_tbl khi chạy lại một phần, NULL = tất cả'),
        ('src_version_from', 'BIGINT', 'last_src_version trước khi chạy'),
        ('src_version_to', 'BIGINT', 'Delta version snapshot đã đọc'),
        ('watermark_from', 'TIMESTAMP', 'watermark_value trước khi chạy'),
        ('watermark_to', 'TIMESTAMP', 'Max watermark_column của phần đã đọc (partner: EventProcessedUtcTime, 3rd-party: crawled_at)'),
        ('overlap_minutes', 'INT', 'Số phút đọc lùi so với watermark_from'),
        ('read_rows', 'BIGINT', 'Số event đọc được'),
        ('valid_rows', 'BIGINT', 'Số event hợp lệ đưa vào xử lý'),
        ('ignored_rows', 'BIGINT', 'Số event bỏ qua (bảng chưa cấu hình, tombstone, truncate...)'),
        ('rejected_rows', 'BIGINT', 'Số event lỗi ghi vào ctrl_cdc_reject'),
        ('ignored_detail', 'STRING', 'JSON số event bỏ qua theo src_object/lý do'),
        ('tbl_total_count', 'INT', 'Số bảng đích trong phạm vi chạy'),
        ('tbl_success_count', 'INT', 'Số bảng SUCCESS'),
        ('tbl_failed_count', 'INT', 'Số bảng FAILED'),
        ('tbl_no_data_count', 'INT', 'Số bảng không có event'),
        ('status', 'STRING', 'RUNNING | SUCCESS | PARTIAL_FAILED | FAILED | NO_DATA | SKIPPED_CONCURRENT | ABANDONED'),
        ('started_at', 'TIMESTAMP', 'Bắt đầu (UTC)'),
        ('ended_at', 'TIMESTAMP', 'Kết thúc (UTC)'),
        ('duration_ms', 'BIGINT', 'Thời gian chạy (ms)'),
        ('run_params', 'STRING', 'JSON tham số notebook'),
        ('error_message', 'STRING', 'Lỗi tổng (tóm tắt)'),
        ('read_mode', 'STRING', 'Cách đọc raw: VERSION | FULL'),
        ('read_note', 'STRING', 'Chi tiết cách đọc: khoảng version, số file, lý do đọc FULL'),
        ('output_versions_json', 'STRING', 'NB_00: version các bảng pl sở hữu khi run SUCCESS / NO_DATA (lát cắt cho luồng khác)'),
     ]},
    {"fq": 'lh_vv_ctrl.dbo.ctrl_log_table_run', "partition": None, "comment": 'Log từng bảng đích trong mỗi lần chạy extract CDC', "props": None,
     "cols": [
        ('exec_id', 'STRING', 'Khớp ctrl_log_run.exec_id'),
        ('run_id', 'STRING', 'RunId của pipeline'),
        ('pl_name', 'STRING', 'Pipeline'),
        ('src_schema', 'STRING', 'Schema bảng nguồn raw'),
        ('src_tbl', 'STRING', 'Bảng nguồn raw'),
        ('src_object', 'STRING', 'Bảng trong Debezium source.table'),
        ('trg_schema', 'STRING', 'Schema bảng đích, vd lh_vv_silver.dbo'),
        ('trg_tbl', 'STRING', 'Bảng đích, vd slv_products'),
        ('priority', 'INT', 'Wave chạy (ctrl_mng_pipeline_config.priority)'),
        ('attempt_no', 'INT', 'Lần thử (1 = lần đầu, >1 = retry)'),
        ('input_rows', 'BIGINT', 'Số event của bảng trong phần đã đọc (3rd-party: số dòng sau khi tách khối/mảng)'),
        ('stale_rows', 'BIGINT', 'Số event cũ hơn state, bị bỏ qua'),
        ('applicable_rows', 'BIGINT', 'Số event mới hơn state, được áp dụng'),
        ('entity_rows', 'BIGINT', 'Số entity (khoá) đưa vào MERGE (3rd-party: số khoá bảng)'),
        ('upsert_event_rows', 'BIGINT', 'Số event c/r/u được áp dụng'),
        ('delete_event_rows', 'BIGINT', 'Số event d được áp dụng'),
        ('inserted_rows', 'BIGINT', 'MERGE: số dòng insert'),
        ('updated_rows', 'BIGINT', 'MERGE: số dòng update (gồm xoá mềm)'),
        ('cast_null_rows', 'BIGINT', 'Số giá trị có ở raw nhưng thành NULL sau chuyển kiểu'),
        ('rejected_rows', 'BIGINT', 'Số event của bảng bị reject'),
        ('status', 'STRING', 'SUCCESS | FAILED | NO_DATA | SKIPPED'),
        ('started_at', 'TIMESTAMP', 'Bắt đầu (UTC)'),
        ('ended_at', 'TIMESTAMP', 'Kết thúc (UTC)'),
        ('duration_ms', 'BIGINT', 'Thời gian xử lý bảng (ms)'),
        ('error_message', 'STRING', 'Lỗi của bảng (tóm tắt)'),
        ('src_versions_json', 'STRING', 'Node: JSON {bảng đầu vào: {v, tid, ts, dirty}} đã đọc'),
        ('trg_version', 'BIGINT', 'Node: version bảng đích sau MERGE'),
        ('deactivated_rows', 'BIGINT', 'Node: số dòng xoá mềm'),
        ('qg_json', 'STRING', 'Node: kết quả exit QG, hộp thư, cảnh báo'),
     ]},
    {"fq": 'lh_vv_ctrl.dbo.ctrl_cdc_state', "partition": ['src_tbl'], "comment": 'State CDC theo entity - đảm bảo chạy lại không áp dụng event cũ', "props": None,
     "cols": [
        ('trg_schema', 'STRING', 'Schema bảng đích (khoá)'),
        ('trg_tbl', 'STRING', 'Bảng đích (khoá)'),
        ('entity_key', 'STRING', 'Giá trị khoá; khoá ghép nối bằng | (khoá). 3rd-party: poi_id cho mọi bảng'),
        ('src_schema', 'STRING', 'Schema bảng nguồn raw'),
        ('src_tbl', 'STRING', 'Bảng nguồn raw'),
        ('src_object', 'STRING', 'Bảng trong Debezium source.table'),
        ('cdc_op', 'STRING', 'Op của event mới nhất: c | r | u | d'),
        ('cdc_ts_ms', 'BIGINT', 'source.ts_ms của event mới nhất (3rd-party: crawled_at tính ms)'),
        ('cdc_lsn', 'BIGINT', 'source.lsn của event mới nhất'),
        ('raw_ingested_at', 'TIMESTAMP', 'EventProcessedUtcTime của event mới nhất (3rd-party: crawled_at)'),
        ('is_deleted', 'BOOLEAN', 'Entity đang ở trạng thái xoá mềm'),
        ('src_version', 'BIGINT', 'Delta version raw của lần áp dụng'),
        ('last_run_id', 'STRING', 'run_id lần cập nhật gần nhất'),
        ('updated_at', 'TIMESTAMP', 'Thời điểm cập nhật state (UTC)'),
     ]},
    {"fq": 'lh_vv_ctrl.dbo.ctrl_cdc_reject', "partition": ['src_tbl'], "comment": 'Event CDC lỗi, không áp dụng được vào bảng đích', "props": None,
     "cols": [
        ('event_hash', 'STRING', 'sha2(source||op||before||after) - khoá MERGE'),
        ('pl_name', 'STRING', 'Pipeline'),
        ('src_schema', 'STRING', 'Schema bảng nguồn raw'),
        ('src_tbl', 'STRING', 'Bảng nguồn raw'),
        ('src_object', 'STRING', 'Bảng trong Debezium source.table'),
        ('src_db', 'STRING', 'Database trong Debezium source.db (3rd-party: source_name)'),
        ('trg_schema', 'STRING', 'Schema bảng đích (nếu xác định được)'),
        ('trg_tbl', 'STRING', 'Bảng đích (nếu xác định được)'),
        ('cdc_op', 'STRING', 'Op của event'),
        ('entity_key', 'STRING', 'Khoá entity (có thể NULL nếu lỗi thiếu khoá)'),
        ('cdc_ts_ms', 'BIGINT', 'source.ts_ms'),
        ('cdc_lsn', 'BIGINT', 'source.lsn'),
        ('raw_ingested_at', 'TIMESTAMP', 'EventProcessedUtcTime'),
        ('reject_reason', 'STRING', 'MISSING_ENTITY_KEY | MISSING_EVENT_ORDER | MISSING_RAW_CURSOR | INVALID_PAYLOAD'),
        ('reject_detail', 'STRING', 'Mô tả chi tiết'),
        ('before_payload', 'STRING', 'Giá trị before gốc'),
        ('after_payload', 'STRING', 'Giá trị after gốc (3rd-party: raw_payload, hoặc phần tử lỗi của 1 bảng)'),
        ('source_payload', 'STRING', 'Giá trị source gốc (3rd-party: JSON event_id, source_name, source_id, language_code, crawled_at)'),
        ('first_run_id', 'STRING', 'run_id lần đầu phát hiện'),
        ('last_run_id', 'STRING', 'run_id lần gần nhất phát hiện'),
        ('reject_count', 'INT', 'Số lần bị đọc lại và reject'),
        ('first_rejected_at', 'TIMESTAMP', 'Lần đầu reject (UTC)'),
        ('last_rejected_at', 'TIMESTAMP', 'Lần gần nhất reject (UTC)'),
        ('is_resolved', 'BOOLEAN', 'Đã xử lý xong (đánh dấu tay hoặc khi nạp lại)'),
        ('resolved_at', 'TIMESTAMP', 'Thời điểm đánh dấu đã xử lý'),
     ]},
    {"fq": 'lh_vv_gold.enr.gld_enr_poi_info', "partition": None, "comment": None, "props": PROPS,
     "cols": [
        ('poi_id', 'STRING NOT NULL', None),
        ('source_name', 'STRING', None),
        ('source_id', 'STRING', None),
        ('language_code', 'STRING', None),
        ('poi_name', 'STRING', None),
        ('poi_name_normalized', 'STRING', None),
        ('types', 'STRING', None),
        ('operating_status', 'STRING', None),
        ('primary_type', 'STRING', None),
        ('website_url', 'STRING', None),
        ('house_number', 'STRING', None),
        ('street', 'STRING', None),
        ('ward', 'STRING', None),
        ('city', 'STRING', None),
        ('full_address', 'STRING', None),
        ('lat', 'DOUBLE', None),
        ('lng', 'DOUBLE', None),
        ('business_sector', 'STRING', None),
        ('business_category', 'STRING', None),
        ('business_category_vi', 'STRING', None),
        ('category_group', 'STRING', None),
        ('range_low_min', 'INT', None),
        ('range_high_min', 'INT', None),
        ('avg_range_min', 'INT', None),
        ('candidate_best_time_code', 'STRING', None),
        ('canonical_candidate', 'STRING', None),
        ('rating', 'DECIMAL(4,2)', None),
        ('user_rating_count', 'BIGINT', None),
        ('price_level', 'INT', None),
        ('opening_hours_text', 'STRING', None),
        ('review_json', 'STRING', None),
        ('time_to_spend', 'STRING', None),
        ('best_time', 'STRING', None),
        ('cost', 'BIGINT', None),
        ('cost_vnd', 'STRING', None),
        ('service_flag', 'STRING', None),
        ('is_enriched_multi_lang', 'BOOLEAN', None),
        ('is_enriched_destination', 'BOOLEAN', None),
        ('created_at', 'TIMESTAMP', None),
        ('updated_at', 'TIMESTAMP', None),
     ]},
    {"fq": 'lh_vv_gold.enr.gld_enr_poi_review', "partition": None, "comment": None, "props": PROPS,
     "cols": [
        ('review_id', 'STRING NOT NULL', None),
        ('poi_id', 'STRING', None),
        ('rating', 'DECIMAL(3,1)', None),
        ('review_text', 'STRING', None),
        ('review_date', 'TIMESTAMP', None),
        ('review_updated_at', 'TIMESTAMP', None),
        ('is_enriched_multi_lang', 'BOOLEAN', None),
        ('created_at', 'TIMESTAMP', None),
        ('updated_at', 'TIMESTAMP', None),
     ]},
    {"fq": 'lh_vv_silver.dbo.ref_dictionary_place_type', "partition": None, "comment": None, "props": PROPS,
     "cols": [
        ('source_name', 'STRING NOT NULL', None),
        ('place_type', 'STRING NOT NULL', None),
        ('business_sector', 'STRING', None),
        ('business_category', 'STRING', None),
        ('business_category_vi', 'STRING', None),
        ('is_active', 'BOOLEAN', None),
        ('updated_at', 'TIMESTAMP', None),
        ('category_group', 'STRING', None),
        ('range_low_min', 'INT', None),
        ('range_high_min', 'INT', None),
        ('avg_range_min', 'INT', None),
        ('candidate_best_time_code', 'STRING', None),
        ('canonical_candidate', 'STRING', None),
     ]},
    {"fq": 'lh_vv_silver.dbo.ref_dictionary_amenities', "partition": None, "comment": None, "props": PROPS,
     "cols": [
        ('source_name', 'STRING NOT NULL', None),
        ('business_sector', 'STRING NOT NULL', None),
        ('amenity_group', 'STRING NOT NULL', None),
        ('amenity_raw', 'STRING NOT NULL', None),
        ('amenity', 'STRING NOT NULL', None),
        ('amenity_vi', 'STRING', None),
        ('service_flags', 'STRING', None),
        ('is_active', 'BOOLEAN', None),
        ('updated_at', 'TIMESTAMP', None),
     ]},
    {"fq": 'lh_vv_silver.dbo.slv_poi_amenity', "partition": None, "comment": None, "props": PROPS,
     "cols": [
        ('poi_id', 'STRING NOT NULL', None),
        ('business_sector', 'STRING NOT NULL', None),
        ('amenity_group', 'STRING NOT NULL', None),
        ('amenity_raw', 'STRING NOT NULL', None),
        ('amenity', 'STRING', None),
        ('amenity_vi', 'STRING', None),
        ('service_flags', 'STRING', None),
        ('row_hash', 'STRING', None),
        ('created_at', 'TIMESTAMP', None),
        ('updated_at', 'TIMESTAMP', None),
        ('deleted_at', 'TIMESTAMP', None),
     ]},
    # [THÊM 22/10] 20261022-enr-landing-and-gld-flags: landing enrich, chỉ DDL.
    {"fq": 'lh_vv_silver.dbo.slv_enr_poi_info_multi_lang', "partition": None, "comment": None, "props": PROPS,
     "cols": [
        ('poi_id', 'STRING NOT NULL', None),
        ('lang', 'STRING', None),
        ('time_to_spend', 'STRING', None),
        ('best_time', 'STRING', None),
        ('cost_vnd', 'STRING', None),
        ('description', 'STRING', None),
        ('created_at', 'TIMESTAMP', None),
        ('updated_at', 'TIMESTAMP', None),
     ]},
    {"fq": 'lh_vv_silver.dbo.slv_enr_poi_destination', "partition": None, "comment": None, "props": PROPS,
     "cols": [
        ('poi_id', 'STRING NOT NULL', None),
        ('street', 'STRING', None),
        ('ward', 'STRING', None),
        ('district', 'STRING', None),
        ('city', 'STRING', None),
        ('full_address', 'STRING', None),
        ('created_at', 'TIMESTAMP', None),
        ('updated_at', 'TIMESTAMP', None),
     ]},
    {"fq": 'lh_vv_silver.dbo.slv_enr_poi_review_multi_lang', "partition": None, "comment": None, "props": PROPS,
     "cols": [
        ('review_id', 'STRING NOT NULL', None),
        ('poi_id', 'STRING', None),
        ('review_text', 'STRING', None),
        ('created_at', 'TIMESTAMP', None),
        ('updated_at', 'TIMESTAMP', None),
     ]},
    # [SỬA 20/10] Bảng node: cột = cột build b1entitybuild + TECH_COLUMNS lib transform (giống ensure_target).
    {"fq": 'lh_vv_silver.dbo.slv_poi_entity_non_enrich', "partition": None, "comment": None, "props": PROPS,
     "cols": [
        ('poi_id', 'STRING', None),
        ('source_name', 'STRING', None),
        ('source_id', 'STRING', None),
        ('language_code', 'STRING', None),
        ('poi_name', 'STRING', None),
        ('poi_name_normalized', 'STRING', None),
        ('types_json', 'STRING', None),
        ('operating_status', 'STRING', None),
        ('primary_type', 'STRING', None),
        ('website_url', 'STRING', None),
        ('house_number', 'STRING', None),
        ('street', 'STRING', None),
        ('ward', 'STRING', None),
        ('city', 'STRING', None),
        ('full_address', 'STRING', None),
        ('lat', 'DOUBLE', None),
        ('lng', 'DOUBLE', None),
        ('business_sector', 'STRING', None),
        ('business_category', 'STRING', None),
        ('business_category_vi', 'STRING', None),
        ('category_group', 'STRING', None),
        ('range_low_min', 'INT', None),
        ('range_high_min', 'INT', None),
        ('avg_range_min', 'INT', None),
        ('candidate_best_time_code', 'STRING', None),
        ('canonical_candidate', 'STRING', None),
        ('row_hash', 'STRING', None),
        ('created_at', 'TIMESTAMP', None),
        ('updated_at', 'TIMESTAMP', None),
        ('deleted_at', 'TIMESTAMP', None),
     ]},
]

def _q(text):
    return text.replace("\\", "\\\\").replace("'", "\\'")


def create_sql(spec):
    parts = []
    for name, dtype, comment in spec["cols"]:
        piece = f"`{name}` {dtype}"
        if comment:
            piece += " COMMENT '" + _q(comment) + "'"
        parts.append(piece)
    sql = "CREATE TABLE IF NOT EXISTS " + spec["fq"] + " (\n    " + ",\n    ".join(parts) + "\n) USING DELTA"
    if spec["partition"]:
        sql += "\nPARTITIONED BY (" + ", ".join(spec["partition"]) + ")"
    if spec["comment"]:
        sql += "\nCOMMENT '" + _q(spec["comment"]) + "'"
    if spec["props"]:
        sql += "\nTBLPROPERTIES (" + spec["props"] + ")"
    return sql


def schema_diff(spec, fields, partition_cols):
    """So theo tên (không theo vị trí): thiếu / thừa / lệch kiểu hoặc nullability / lệch partition."""
    want = {n.lower(): (t.replace(" NOT NULL", "").lower(), "NOT NULL" not in t) for n, t, _c in spec["cols"]}
    got = {f.name.lower(): (f.dataType.simpleString(), f.nullable) for f in fields}
    issues = [f"thiếu {n}" for n in want if n not in got]
    issues += [f"thừa {n}" for n in got if n not in want]
    issues += [f"{n}: có {got[n]}, cần {want[n]}" for n in want if n in got and got[n] != want[n]]
    if [p.lower() for p in partition_cols] != [p.lower() for p in (spec["partition"] or [])]:
        issues.append(f"partition: có {list(partition_cols)}, cần {spec['partition']}")
    return issues
# END_INIT_TABLE_SPECS


In [ ]:


TABLE_CONFIGS = [
    {
        "source_table": "products",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_products",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "string"},
            "external_id": {"path": "external_id", "type": "string"},
            "product_line": {"path": "product_line", "type": "string"},
            "status": {"path": "status", "type": "string"},
            "attributes": {"path": "attributes", "type": "string"},
            "metadata": {"path": "metadata", "type": "string"},
            "raw_sync_data": {"path": "raw_sync_data", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "deleted_at": {"path": "deleted_at", "type": "string"},
        },
        "toast_columns": ["attributes", "metadata", "raw_sync_data"],
    },
    {
        "source_table": "product_translations",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_translations",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "bigint"},
            "product_id": {"path": "product_id", "type": "string"},
            "locale": {"path": "locale", "type": "string"},
            "name": {"path": "name", "type": "string"},
            "metadata": {"path": "metadata", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
        },
        "toast_columns": ["metadata"],
    },
    {
        "source_table": "product_attributes",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_attributes",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "string"},
            "external_id": {"path": "external_id", "type": "string"},
            "code": {"path": "code", "type": "string"},
            "status": {"path": "status", "type": "string"},
            "product_category_scope": {"path": "product_category_scope", "type": "string"},
            "raw_sync_data": {"path": "raw_sync_data", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
        },
        "toast_columns": ["raw_sync_data"],
    },
    {
        "source_table": "product_attribute_translations",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_attribute_translations",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "bigint"},
            "attribute_id": {"path": "attribute_id", "type": "string"},
            "locale": {"path": "locale", "type": "string"},
            "name": {"path": "name", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
        },
        "toast_columns": [],
    },
    {
        "source_table": "product_attribute_values",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_attribute_values",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "string"},
            "attribute_id": {"path": "attribute_id", "type": "string"},
            "external_id": {"path": "external_id", "type": "string"},
            "code": {"path": "code", "type": "string"},
            "raw_sync_data": {"path": "raw_sync_data", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
        },
        "toast_columns": ["raw_sync_data"],
    },
    {
        "source_table": "product_attribute_value_translations",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_attribute_value_translations",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "bigint"},
            "attribute_id": {"path": "attribute_id", "type": "string"},
            "locale": {"path": "locale", "type": "string"},
            "name": {"path": "name", "type": "string"},
        },
        "toast_columns": [],
    },
    {
        "source_table": "product_variants",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_variants",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "string"},
            "product_id": {"path": "product_id", "type": "string"},
            "status": {"path": "status", "type": "string"},
            "external_id": {"path": "external_id", "type": "string"},
            "attributes": {"path": "attributes", "type": "string"},
            "raw_sync_data": {"path": "raw_sync_data", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "deleted_at": {"path": "deleted_at", "type": "string"},
        },
        "toast_columns": ["attributes", "raw_sync_data"],
    },
    {
        "source_table": "product_variant_translations",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_variant_translations",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "bigint"},
            "variant_id": {"path": "variant_id", "type": "string"},
            "locale": {"path": "locale", "type": "string"},
            "name": {"path": "name", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
        },
        "toast_columns": [],
    },
    {
        "source_table": "product_business_service_relation",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_business_service_relation",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "bigint"},
            "product_id": {"path": "product_id", "type": "string"},
            "business_service_id": {"path": "business_service_id", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
        },
        "toast_columns": [],
    },
    {
        "source_table": "product_posts",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_posts",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "string"},
            "business_service_id": {"path": "business_service_id", "type": "string"},
            "created_by": {"path": "created_by", "type": "string"},
            "updated_by": {"path": "updated_by", "type": "string"},
            "post_code": {"path": "post_code", "type": "string"},
            "external_id": {"path": "external_id", "type": "string"},
            "status": {"path": "status", "type": "string"},
            "original_price": {"path": "original_price", "type": "string"},
            "markup_percent": {"path": "markup_percent", "type": "string"},
            "marketing_price": {"path": "marketing_price", "type": "string"},
            "thumbnail_url": {"path": "thumbnail_url", "type": "string"},
            "gallery": {"path": "gallery", "type": "string"},
            "product_list": {"path": "product_list", "type": "string"},
            "raw_sync_data": {"path": "raw_sync_data", "type": "string"},
            "validity_from": {"path": "validity_from", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "validity_to": {"path": "validity_to", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "deleted_at": {"path": "deleted_at", "type": "string"},
        },
        "toast_columns": ["original_price", "markup_percent", "marketing_price", "gallery", "product_list", "raw_sync_data"],
    },
    {
        "source_table": "product_post_translations",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_product_post_translations",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "bigint"},
            "product_post_id": {"path": "product_post_id", "type": "string"},
            "locale": {"path": "locale", "type": "string"},
            "title": {"path": "title", "type": "string"},
            "short_description": {"path": "short_description", "type": "string"},
            "tags": {"path": "tags", "type": "string"},
            "filter_tags": {"path": "filter_tags", "type": "string"},
            "ai_tags": {"path": "ai_tags", "type": "string"},
            "faqs": {"path": "faqs", "type": "string"},
            "sections": {"path": "sections", "type": "string"},
            "meta_title": {"path": "meta_title", "type": "string"},
            "meta_description": {"path": "meta_description", "type": "string"},
            "slug": {"path": "slug", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
        },
        "toast_columns": ["tags", "filter_tags", "ai_tags", "faqs", "sections"],
    },
    {
        "source_table": "partners",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_partners",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "string"},
            "code": {"path": "code", "type": "string"},
            "lead_id": {"path": "lead_id", "type": "string"},
            "company_name": {"path": "company_name", "type": "string"},
            "company_short_name": {"path": "company_short_name", "type": "string"},
            "company_legal_name": {"path": "company_legal_name", "type": "string"},
            "company_legal_vietnamese_name": {"path": "company_legal_vietnamese_name", "type": "string"},
            "company_tax_id": {"path": "company_tax_id", "type": "string"},
            "company_registration_number": {"path": "company_registration_number", "type": "string"},
            "company_address": {"path": "company_address", "type": "string"},
            "business_sector": {"path": "business_sector", "type": "string"},
            "business_sector_detail": {"path": "business_sector_detail", "type": "string"},
            "detailed_profile": {"path": "detailed_profile", "type": "string"},
            "status": {"path": "status", "type": "string"},
            "assignee_username": {"path": "assignee_username", "type": "string"},
            "operation_status": {"path": "operation_status", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "last_updated_at": {"path": "last_updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "created_by": {"path": "created_by", "type": "string"},
            "last_updated_by": {"path": "last_updated_by", "type": "string"},
            "tier": {"path": "tier", "type": "smallint"},
            "metadata": {"path": "metadata", "type": "string"},
            "parent_partner_id": {"path": "parent_partner_id", "type": "string"},
            "type": {"path": "type", "type": "string"},
            "business_scenario": {"path": "business_scenario", "type": "string"},
            "contact_information": {"path": "contact_information", "type": "string"},
            "business_registration_link": {"path": "business_registration_link", "type": "string"},
            "legal_representative": {"path": "legal_representative", "type": "string"},
            "business_registration_change": {"path": "business_registration_change", "type": "string"},
            "headquarter_information": {"path": "headquarter_information", "type": "string"},
        },
        "toast_columns": ["detailed_profile", "metadata", "contact_information", "legal_representative", "business_registration_change", "headquarter_information"],
    },
    {
        "source_table": "business_services",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_business_services",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "string"},
            "partner_id": {"path": "partner_id", "type": "string"},
            "business_location_id": {"path": "business_location_id", "type": "string"},
            "name": {"path": "name", "type": "string"},
            "phone": {"path": "phone", "type": "string"},
            "hotline": {"path": "hotline", "type": "string"},
            "email": {"path": "email", "type": "string"},
            "business_sector": {"path": "business_sector", "type": "string"},
            "operation_status": {"path": "operation_status", "type": "string"},
            "address": {"path": "address", "type": "string"},
            "direction_note": {"path": "direction_note", "type": "string"},
            "bio": {"path": "bio", "type": "string"},
            "story": {"path": "story", "type": "string"},
            "usps": {"path": "usps", "type": "string"},
            "local_tips": {"path": "local_tips", "type": "string"},
            "media": {"path": "media", "type": "string"},
            "is_main": {"path": "is_main", "type": "boolean"},
            "amentities": {"path": "amentities", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "last_updated_at": {"path": "last_updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "created_by": {"path": "created_by", "type": "string"},
            "last_updated_by": {"path": "last_updated_by", "type": "string"},
            "metadata": {"path": "metadata", "type": "string"},
            "is_active": {"path": "is_active", "type": "boolean"},
            "last_in_activated_at": {"path": "last_in_activated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "last_in_activated_by": {"path": "last_in_activated_by", "type": "string"},
            "website_urls": {"path": "website_urls", "type": "string"},
            "last_verify_at": {"path": "last_verify_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
        },
        "toast_columns": ["address", "direction_note", "bio", "story", "usps", "local_tips", "media", "amentities", "metadata", "website_urls"],
    },
    {
        "source_table": "business_locations",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_business_locations",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "string"},
            "name": {"path": "name", "type": "string"},
            "type": {"path": "type", "type": "string"},
            "address": {"path": "address", "type": "string"},
            "direction_note": {"path": "direction_note", "type": "string"},
            "partner_id": {"path": "partner_id", "type": "string"},
            "business_scenario": {"path": "business_scenario", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "last_updated_at": {"path": "last_updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "last_updated_by": {"path": "last_updated_by", "type": "string"},
            "metadata": {"path": "metadata", "type": "string"},
            "created_by": {"path": "created_by", "type": "string"},
        },
        "toast_columns": ["address", "direction_note", "metadata"],
    },
    {
        "source_table": "hotel_room_types",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_hotel_room_types",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "string"},
            "code": {"path": "code", "type": "string"},
            "name": {"path": "name", "type": "string"},
            "description": {"path": "description", "type": "string"},
            "status": {"path": "status", "type": "string"},
            "max_occupancy": {"path": "max_occupancy", "type": "int"},
            "adult_capacity": {"path": "adult_capacity", "type": "int"},
            "child_capacity": {"path": "child_capacity", "type": "int"},
            "max_extra_bed": {"path": "max_extra_bed", "type": "int"},
            "max_child_surcharge": {"path": "max_child_surcharge", "type": "int"},
            "room_size": {"path": "room_size", "type": "int"},
            "view_type": {"path": "view_type", "type": "string"},
            "total_physical_rooms": {"path": "total_physical_rooms", "type": "int"},
            "total_pseudo_rooms": {"path": "total_pseudo_rooms", "type": "int"},
            "out_of_service_count": {"path": "out_of_service_count", "type": "int"},
            "out_of_order_count": {"path": "out_of_order_count", "type": "int"},
            "total_rooms": {"path": "total_rooms", "type": "int"},
            "available_rooms": {"path": "available_rooms", "type": "int"},
            "amenities": {"path": "amenities", "type": "string"},
            "media_files": {"path": "media_files", "type": "string"},
            "tags": {"path": "tags", "type": "string"},
            "business_service_id": {"path": "business_service_id", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "deleted_at": {"path": "deleted_at", "type": "string"},
        },
        "toast_columns": ["description", "amenities", "media_files", "tags"],
    },
    {
        "source_table": "hotel_rate_plans",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_hotel_rate_plans",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "string"},
            "business_service_id": {"path": "business_service_id", "type": "string"},
            "code": {"path": "code", "type": "string"},
            "name": {"path": "name", "type": "string"},
            "description": {"path": "description", "type": "string"},
            "status": {"path": "status", "type": "string"},
            "kind": {"path": "kind", "type": "string"},
            "meal_plan": {"path": "meal_plan", "type": "int"},
            "include_tax": {"path": "include_tax", "type": "boolean"},
            "include_service_charge": {"path": "include_service_charge", "type": "boolean"},
            "is_derived": {"path": "is_derived", "type": "boolean"},
            "valid_from": {"path": "valid_from", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "valid_to": {"path": "valid_to", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "is_visible_on_website": {"path": "is_visible_on_website", "type": "boolean"},
            "tags": {"path": "tags", "type": "string"},
            "cancellation_policies": {"path": "cancellation_policies", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "deleted_at": {"path": "deleted_at", "type": "string"},
        },
        "toast_columns": ["description", "tags", "cancellation_policies"],
    },
    {
        "source_table": "business_service_i18ns",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_business_service_i18ns",
        "key": ["business_service_id", "locale"],
        "columns": {
            "business_service_id": {"path": "business_service_id", "type": "string"},
            "locale": {"path": "locale", "type": "string"},
            "name": {"path": "name", "type": "string"},
            "bio": {"path": "bio", "type": "string"},
            "story": {"path": "story", "type": "string"},
            "direction_note": {"path": "direction_note", "type": "string"},
            "usps": {"path": "usps", "type": "string"},
            "local_tips": {"path": "local_tips", "type": "string"},
            "amentities": {"path": "amentities", "type": "string"},
            "media": {"path": "media", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
        },
        "toast_columns": ["name", "bio", "story", "direction_note", "usps", "local_tips", "amentities", "media"],
    },
    {
        "source_table": "business_service_poi_link",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_business_service_poi_link",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "bigint"},
            "business_service_id": {"path": "business_service_id", "type": "string"},
            "poi_id": {"path": "poi_id", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_S_TS"},
            "lang": {"path": "lang", "type": "string"},
        },
        "toast_columns": [],
    },
    {
        "source_table": "orders",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_orders",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "string"},
            "code": {"path": "code", "type": "string"},
            "contact_info": {"path": "contact_info", "type": "string"},
            "user_id": {"path": "user_id", "type": "string"},
            "total_payment": {"path": "total_payment", "type": "debezium_decimal(19,4)"},
            "currency": {"path": "currency", "type": "string"},
            "status": {"path": "status", "type": "string"},
            "payment_status": {"path": "payment_status", "type": "string"},
            "payment_id": {"path": "payment_id", "type": "string"},
            "metadata": {"path": "metadata", "type": "string"},
            "expire_at": {"path": "expire_at", "type": "timestamp", "rule": "EPOCH_MS_TS"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_MS_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_MS_TS"},
        },
        "toast_columns": ["contact_info", "metadata"],
    },
    {
        "source_table": "order_refs",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_order_refs",
        "key": "id",
        "columns": {
            "id": {"path": "id", "type": "string"},
            "order_id": {"path": "order_id", "type": "string"},
            "status": {"path": "status", "type": "string"},
            "total_payment": {"path": "total_payment", "type": "debezium_decimal(19,4)"},
            "sub_total": {"path": "sub_total", "type": "debezium_decimal(19,4)"},
            "external_order_id": {"path": "external_order_id", "type": "string"},
            "internal_input_object": {"path": "internal_input_object", "type": "string"},
            "product_type": {"path": "product_type", "type": "string"},
            "raw_data": {"path": "raw_data", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_MS_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_MS_TS"},
        },
        "toast_columns": ["internal_input_object", "raw_data"],
    },
    {
        "source_table": "order_item_tickets",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_order_item_tickets",
        "key": "order_ref_id",
        "columns": {
            "order_id": {"path": "order_id", "type": "string"},
            "order_ref_id": {"path": "order_ref_id", "type": "string"},
            "product_id": {"path": "product_id", "type": "string"},
            "variant_id": {"path": "variant_id", "type": "string"},
            "provider": {"path": "provider", "type": "string"},
            "pnr": {"path": "pnr", "type": "string"},
            "ticket_number": {"path": "ticket_number", "type": "string"},
            "verify_code": {"path": "verify_code", "type": "string"},
            "external_ticket_product_name": {"path": "external_ticket_product_name", "type": "string"},
            "usage_date": {"path": "usage_date", "type": "timestamp", "rule": "ISO_UTC_TS"},
            "valid_from": {"path": "valid_from", "type": "timestamp", "rule": "ISO_UTC_TS"},
            "valid_to": {"path": "valid_to", "type": "timestamp", "rule": "ISO_UTC_TS"},
            "raw_data": {"path": "raw_data", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_MS_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_MS_TS"},
        },
        "toast_columns": ["raw_data"],
    },
    {
        "source_table": "order_item_hotels",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_order_item_hotels",
        "key": "order_id",
        "columns": {
            "order_id": {"path": "order_id", "type": "string"},
            "order_ref_id": {"path": "order_ref_id", "type": "string"},
            "check_in_date": {"path": "check_in_date", "type": "timestamp", "rule": "ISO_UTC_TS"},
            "checkout_date": {"path": "checkout_date", "type": "timestamp", "rule": "ISO_UTC_TS"},
            "number_of_room": {"path": "number_of_room", "type": "smallint"},
            "hotel_id": {"path": "hotel_id", "type": "string"},
            "pnr": {"path": "pnr", "type": "string"},
            "room_type_id": {"path": "room_type_id", "type": "string"},
            "rate_plan_id": {"path": "rate_plan_id", "type": "string"},
            "raw_data": {"path": "raw_data", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_MS_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_MS_TS"},
        },
        "toast_columns": ["raw_data"],
    },
    {
        "source_table": "order_item_flights",
        "source_schema": "public",
        "target_schema": "lh_vv_silver.dbo",
        "target_table": "slv_pn_order_item_flights",
        "key": "order_id",
        "columns": {
            "order_id": {"path": "order_id", "type": "string"},
            "order_ref_id": {"path": "order_ref_id", "type": "string"},
            "provider": {"path": "provider", "type": "string"},
            "pnr": {"path": "pnr", "type": "string"},
            "booking_id": {"path": "booking_id", "type": "string"},
            "segment_id": {"path": "segment_id", "type": "smallint"},
            "carrier_code": {"path": "carrier_code", "type": "string"},
            "carrier_name": {"path": "carrier_name", "type": "string"},
            "departure": {"path": "departure", "type": "string"},
            "departure_time": {"path": "departure_time", "type": "timestamp", "rule": "ISO_UTC_TS"},
            "departure_terminal": {"path": "departure_terminal", "type": "string"},
            "arrival": {"path": "arrival", "type": "string"},
            "arrival_time": {"path": "arrival_time", "type": "timestamp", "rule": "ISO_UTC_TS"},
            "arrival_terminal": {"path": "arrival_terminal", "type": "string"},
            "flight_number": {"path": "flight_number", "type": "string"},
            "booking_class": {"path": "booking_class", "type": "string"},
            "aircraft": {"path": "aircraft", "type": "string"},
            "fare_basis_code": {"path": "fare_basis_code", "type": "string"},
            "fare_basis_detail": {"path": "fare_basis_detail", "type": "string"},
            "raw_data": {"path": "raw_data", "type": "string"},
            "src_created_at": {"path": "created_at", "type": "timestamp", "rule": "EPOCH_MS_TS"},
            "src_updated_at": {"path": "updated_at", "type": "timestamp", "rule": "EPOCH_MS_TS"},
        },
        "toast_columns": ["fare_basis_detail", "raw_data"],
    },
]


In [ ]:
# [THÊM 03/10 - 3P]
# =============================================================================
# [THÊM 03/10] TABLE_CONFIGS_3P — bảng slv_3p_poi_* của nguồn 3rd-party (lh_vv_bronze.dbo.brz_3rd_crawler_poi_stream)
# Notebook xử lý: NB_EXTRACT_3RD_PARTY_CDC_BRZ_TO_SLV. Thiết kế: claude/POI_3P_EXTRACT_REDESIGN.md §5.4, §5.8
#
# [SỬA 04/10 - 3P] Đối chiếu payload thật (truy vấn A, E; VERIFY_3P_CONFIG_0410 V1–V13 trên 21.168 dòng raw, crawl
#   tới 16/09) + quyết định của anh Hòa 04/10. 14 bảng (bảng policy tắt ở pipeline_config).
#   - Cột = khoá có trong payload. Bỏ cột của khoá không tồn tại (V2: dữ liệu bảng cũ toàn NULL, trừ primary_type — xem D2).
#   - D1  Khoá luôn null (V1) -> "active": False: registry is_active = false, notebook bỏ qua, bảng đích không có cột.
#         Collector bắt đầu điền: UPDATE registry is_active = true rồi full_reload bảng đó. Bảng policy (khối luôn null):
#         tắt ở ctrl_mng_pipeline_config.is_active = 0.
#   - D2  Bỏ cột JSON nguyên khối trùng các cột đã tách (address_json, contact_json, amenity_json, opening_hours_json,
#         rating_json). Giữ nguyên JSON, transform sau: raw_data_json, amenity_schema_json, ext_attributes_json,
#         facilities_json, secondary_hours_json (anh Hòa chốt 04/10 15:53: amenity_schema / ext_attributes không tách
#         key-value ở extract -> không cần load_mode DOC_MAP, code lib / notebook giữ nguyên bản 04/10 sáng).
#   - D3  [SỬA 09/10] open_now của poi_opening_hours thành cột boolean. ext_attributes.open_now vẫn nằm trong JSON.
#   - D4  Content: khoá (poi_id, locale, content_type), trùng khoá lấy phần tử đầu (V8: 29 nhóm trùng, 0 nhóm khác nội dung).
#   - D5  Bỏ cột trùng 100% với cột khác (V4, V5, V12): slv_3p_poi.price_level (= poi_price.price_level),
#         slv_3p_poi.country_code (= poi_address.country_code), slv_3p_poi.source (= source_name viết hoa),
#         address.address_google (= full_address), enrichment.time_to_spend (= time_to_spend_minutes). Giữ enrichment_source_id.
#   - D7  Mỗi cột có comment ý nghĩa. "(nghĩa chưa xác nhận)": suy từ tên / mẫu, chưa có tài liệu collector.
#   - Mảng 1 tầng để cột JSON (anh Hòa chốt 04/10 15:53): periods_json, experiences_json, types_json,
#     weekday_text_json. [BỎ 19/10] subcategory_tags_json đã ra khỏi registry.
#     Cần tách sau: chỉ thêm cấu hình DOC_ARRAY / LANG_ARRAY, không sửa code.
#   Từng chỗ: [THÊM 04/10] / [BỎ 04/10] / [TẮT 04/10 D1] / [SỬA 04/10].
#
# [SỬA 09/10] Payload raw, probe E1 (33/33 tài liệu, extra_info rỗng, không enrichment).
#   L1 chỉ extract 1:1 và cast kiểu. Quyết định nghiệp vụ ở L2.
#   Bỏ AMENITY_FLAGS, available_langs_json, enrichment_failed_langs_json.
#   Bỏ khỏi registry: slv_3p_poi_price, slv_3p_poi_raw_data, slv_3p_poi_content,
#   slv_3p_poi_enrichment, slv_3p_poi_review_i18n. slv_3p_poi_policy vẫn tắt ở pipeline_config.
#   Thêm lại slv_3p_poi_price (10/10, 13/53 tài liệu là object đủ 4 khóa). Kỳ vọng: 10 bảng, 98 cột, 15 cột tắt.
#
# Mỗi bảng:
#   source_table : src_object = khối trong payload ("$" = gốc payload; bảng LANG*: đường dẫn trong object bản địa hoá)
#   (load_mode / align_path / dedup_order / is_active của bảng: cell ctrl_mng_pipeline_config, mục 3)
#   key          : cột khoá (str hoặc list theo thứ tự)
#   columns      : {tên cột: {"path": json_path, "type": kiểu Spark, "rule": convert_rule (mặc định NONE),
#                             "active": False nếu tắt (mặc định bật)}}
#
# json_path (bảng ở NB_LIB_EXTRACT_RAWDATA §1):
#   a.b           trường trong phần tử / khối (bảng DOC "$": trong gốc payload)
#   $             cả phần tử / khối dạng chuỗi JSON
#   _doc.<cột>    cột của bảng raw            _pos    vị trí phần tử (từ 0)
#   [BỎ 09/10] _lang, _langs, _align: 3P không còn bảng LANG / LANG_ARRAY.
#   p1,p2         nhiều phần, chỉ với rule HASH_*
#
# Quy ước: cột chứa JSON (object/mảng) giữ dạng chuỗi, tên có hậu tố _json (quyết định mở #4: JSON STRING).
#          Tên cột giữ theo tên trường payload (snake_case), trừ trường camelCase của enrichment.
# Kiểu: V1 04/10 CAST mọi cột 0 lỗi (thời gian ISO 8601 9 chữ số lẻ cắt về microgiây).
# =============================================================================
_S = "lh_vv_silver.dbo"
# Khoá địa điểm: md5(source_name + source_id) dạng UUID = poi_group_id của luồng cũ (quyết định 03/10)
POI_ID = {"path": "_doc.source_name,_doc.source_id", "type": "string", "rule": "HASH_MD5_UUID"}


def _str(path):
    return {"path": path, "type": "string"}


def _bool(path):
    return {"path": path, "type": "boolean"}


def _off(col):
    """[THÊM 04/10 D1] Cột tắt (registry is_active = false): khoá có trong payload nhưng luôn null (V1 04/10)."""
    return {**col, "active": False}


# [BỎ 09/10] AMENITY_FLAGS / AMENITY_FLAGS_OFF: E1 amenity_flat_flag_docs = 0.
#   Map ext_attributes sang cờ nghiệp vụ là logic L2, không làm ở extract.

# [SỬA 14/10] TABLE_CONFIGS_3P đọc normalized_payload. 9 bảng, 53 cột, 0 cột tắt.
# [BỎ 19/10] Bỏ slv_3p_poi.subcategory_tags_json và slv_3p_poi_address.state_province.
# Đối số rule nằm ở "arg". Không hard-code tên khóa Google trong notebook extract.
TABLE_CONFIGS_3P = [
    {
        "source_table": "$", "target_schema": _S, "target_table": "slv_3p_poi",
        "key": "poi_id",
        "columns": {
            "poi_id": POI_ID,
            "source_name": _str("_doc.source_name"),
            "source_id": _str("_doc.source_id"),
            "language_code": {"path": "_doc.language_code", "type": "string", "rule": "LOWER_TRIM"},
            "poi_name": _str("displayName"),
            "poi_name_normalized": {"path": "displayName", "type": "string", "rule": "LOWER_TRIM"},
            "types_json": _str("types"),
            "operating_status": {"path": "businessStatus", "type": "string", "rule": "LOWER_TRIM"},
            "primary_type": _str("primaryType"),
        },
    },
    {
        "source_table": "$", "target_schema": _S, "target_table": "slv_3p_poi_address",
        "key": "poi_id",
        "columns": {
            "poi_id": POI_ID,
            "house_number": {"path": "addressComponents", "type": "string", "rule": "ARRAY_PICK", "arg": {'match': 'types', 'values': ['street_number'], 'take': 'longText'}},
            "street": {"path": "addressComponents", "type": "string", "rule": "ARRAY_PICK", "arg": {'match': 'types', 'values': ['route'], 'take': 'longText'}},
            "ward": {"path": "addressComponents", "type": "string", "rule": "ARRAY_PICK", "arg": {'match': 'types', 'values': ['administrative_area_level_2'], 'take': 'longText'}},
            "city": {"path": "addressComponents", "type": "string", "rule": "ARRAY_PICK", "arg": {'match': 'types', 'values': ['administrative_area_level_1'], 'take': 'longText'}},
            "country_code": {"path": "addressComponents", "type": "string", "rule": "ARRAY_PICK", "arg": {'match': 'types', 'values': ['country'], 'take': 'shortText'}},
            "postal_code": {"path": "addressComponents", "type": "string", "rule": "ARRAY_PICK", "arg": {'match': 'types', 'values': ['postal_code'], 'take': 'longText'}},
            "full_address": _str("formattedAddress"),
            "lat": {"path": "location.latitude", "type": "double"},
            "lng": {"path": "location.longitude", "type": "double"},
            "plus_code": _str("plusCode.globalCode"),
        },
    },
    {
        "source_table": "$", "target_schema": _S, "target_table": "slv_3p_poi_contact",
        "key": "poi_id",
        "columns": {
            "poi_id": POI_ID,
            "website_url": _str("websiteUri"),
            "google_maps_url": _str("googleMapsUri"),
            "phone": _str("nationalPhoneNumber"),
        },
    },
    {
        "source_table": "$", "target_schema": _S, "target_table": "slv_3p_poi_price",
        "key": "poi_id",
        "columns": {
            "poi_id": POI_ID,
            "currency": _str("priceRange.startPrice.currencyCode"),
            "price_level": {"path": "priceLevel", "type": "int", "rule": "ENUM_MAP", "arg": {'map': {'PRICE_LEVEL_FREE': 0, 'PRICE_LEVEL_INEXPENSIVE': 1, 'PRICE_LEVEL_MODERATE': 2, 'PRICE_LEVEL_EXPENSIVE': 3, 'PRICE_LEVEL_VERY_EXPENSIVE': 4}}},
            "price_min": {"path": "priceRange.startPrice.units", "type": "decimal(18,2)"},
            "price_max": {"path": "priceRange.endPrice.units", "type": "decimal(18,2)"},
        },
    },
    {
        "source_table": "$", "target_schema": _S, "target_table": "slv_3p_poi_opening_hours",
        "key": "poi_id",
        "columns": {
            "poi_id": POI_ID,
            "periods_json": {"path": "regularOpeningHours.periods", "type": "string", "rule": "ARRAY_PROJECT", "arg": {'fields': [{'name': 'close_hour', 'path': 'close.hour', 'type': 'int'}, {'name': 'close_day', 'path': 'close.day', 'type': 'int'}, {'name': 'open_hour', 'path': 'open.hour', 'type': 'int'}, {'name': 'close_minute', 'path': 'close.minute', 'type': 'int'}, {'name': 'open_minute', 'path': 'open.minute', 'type': 'int'}, {'name': 'day_of_week', 'path': 'open.day', 'type': 'int'}], 'keep_nulls': False}},
            "weekday_text_json": _str("regularOpeningHours.weekdayDescriptions"),
            "open_now": {"path": "currentOpeningHours.openNow,regularOpeningHours.openNow", "type": "boolean", "rule": "COALESCE"},
        },
    },
    {
        "source_table": "$", "target_schema": _S, "target_table": "slv_3p_poi_rating",
        "key": "poi_id",
        "columns": {
            "poi_id": POI_ID,
            "rating_overall": {"path": "rating", "type": "decimal(4,2)"},
            "rating_count": {"path": "userRatingCount", "type": "bigint"},
        },
    },
    {
        "source_table": "$", "target_schema": _S, "target_table": "slv_3p_poi_amenity",
        "key": "poi_id",
        "columns": {
            "poi_id": POI_ID,
            "source_flags_json": {"path": 'servesBreakfast,servesBrunch,servesLunch,servesDinner,servesBeer,servesWine,servesCocktails,servesDessert,servesCoffee,servesVegetarianFood,allowsDogs,goodForChildren,goodForGroups,goodForWatchingSports,liveMusic,menuForChildren,outdoorSeating,restroom,reservable,takeout,delivery,dineIn,curbsidePickup,parkingOptions,paymentOptions,accessibilityOptions,evChargeOptions,fuelOptions', "type": "string", "rule": "JSON_PICK", "arg": {'members': [{'name': 'flags', 'kind': 'boolean', 'keys': ['servesBreakfast', 'servesBrunch', 'servesLunch', 'servesDinner', 'servesBeer', 'servesWine', 'servesCocktails', 'servesDessert', 'servesCoffee', 'servesVegetarianFood', 'allowsDogs', 'goodForChildren', 'goodForGroups', 'goodForWatchingSports', 'liveMusic', 'menuForChildren', 'outdoorSeating', 'restroom', 'reservable', 'takeout', 'delivery', 'dineIn', 'curbsidePickup']}, {'key': 'parkingOptions', 'kind': 'object'}, {'key': 'paymentOptions', 'kind': 'object'}, {'key': 'accessibilityOptions', 'kind': 'object'}, {'key': 'evChargeOptions', 'kind': 'object'}, {'key': 'fuelOptions', 'kind': 'object'}]}},
        },
    },
    {
        "source_table": "reviews", "target_schema": _S, "target_table": "slv_3p_poi_review",
        "key": "review_id",
        "columns": {
            "review_id": {"path": "_doc.source_name,_doc.source_id,authorAttribution.displayName,publishTime", "type": "string", "rule": "HASH_SHA256"},
            "poi_id": POI_ID,
            "author_name": _str("authorAttribution.displayName"),
            "rating": {"path": "rating", "type": "decimal(4,2)"},
            "text": _str("text.text"),
            "time": _str("publishTime"),
            "published_at": {"path": "publishTime", "type": "timestamp", "rule": "ISO_UTC_TS"},
        },
    },
    {
        "source_table": "photos", "target_schema": _S, "target_table": "slv_3p_poi_media",
        "key": "media_dedup_key",
        "columns": {
            "media_dedup_key": {"path": "_doc.source_name,_doc.source_id,name", "type": "string", "rule": "HASH_SHA256_PIPE"},
            "poi_id": POI_ID,
            "photo_api_uri": _str("name"),
            "display_order": {"path": "_pos", "type": "int", "rule": "POS_1BASED"},
            "width_px": {"path": "widthPx", "type": "int"},
            "height_px": {"path": "heightPx", "type": "int"},
            "blob_url": _str("blob_url"),
            "photographer": {"path": "authorAttributions", "type": "string", "rule": "ARRAY_AT", "arg": {"index": 0, "take": "displayName"}},
        },
    },
]


In [ ]:
# BEGIN_REGISTRY_PURE
import re

# Kiểu giả trong TABLE_CONFIGS -> (kiểu Spark thật, convert_rule)
PSEUDO_TYPES = {
    "epoch_seconds_timestamp": ("string", "EPOCH_S_TS"),
}
# Decimal Debezium (decimal.handling.mode = precise): JSON là base64 của số nguyên chưa chia scale
DEBEZIUM_DECIMAL_RE = re.compile(r"debezium_decimal\(\s*(\d+)\s*,\s*(\d+)\s*\)", re.I)


def resolve_type(t: str) -> tuple:
    """Kiểu trong TABLE_CONFIGS -> (data_type, convert_rule).
    "debezium_decimal(19,4)"  -> ("decimal(19,4)", "DECIMAL_BASE64")
    "epoch_seconds_timestamp" -> ("string", "EPOCH_S_TS")
    kiểu khác                 -> (t, "NONE")
    """
    t = t.strip()
    m = DEBEZIUM_DECIMAL_RE.fullmatch(t)
    if m:
        p, s = int(m.group(1)), int(m.group(2))
        if not (0 < p <= 38 and 0 <= s <= p):
            raise ValueError(f"debezium_decimal sai precision/scale: {t}")
        return f"decimal({p},{s})", "DECIMAL_BASE64"
    return PSEUDO_TYPES.get(t, (t, "NONE"))


KNOWN_RULES = {"NONE", "EPOCH_S_TS", "EPOCH_MS_TS", "EPOCH_US_TS", "ISO_UTC_TS", "DATE_DAYS", "DECIMAL_BASE64", "LOWER_TRIM",
               "HASH_MD5_UUID", "HASH_SHA256", "HASH_SHA256_PIPE",
               "COALESCE", "ARRAY_PICK", "ARRAY_AT", "ENUM_MAP", "ARRAY_PROJECT", "POS_1BASED", "JSON_PICK"}



def flatten_table_configs(configs: list, src_schema: str, src_tbl: str) -> list:
    """TABLE_CONFIGS (1 phần tử / bảng) -> danh sách dòng (1 dòng / cột đích)."""
    rows = []
    for cfg in configs:
        keys = cfg["key"] if isinstance(cfg["key"], list) else [cfg["key"]]
        toast_cols = set(cfg.get("toast_columns", []))

        for col_order, (col_name, col_cfg) in enumerate(cfg["columns"].items(), start=1):
            data_type, convert_rule = resolve_type(col_cfg["type"])
            convert_rule = col_cfg.get("rule", convert_rule)
            rows.append({
                "src_schema":        src_schema,
                "src_tbl":           src_tbl,
                "src_object_schema": cfg.get("source_schema"),
                "src_object":        cfg["source_table"],
                "trg_schema":        cfg["target_schema"],
                "trg_tbl":           cfg["target_table"],
                "trg_column":        col_name,
                "json_path":         col_cfg.get("path"),
                "data_type":         data_type,
                "convert_rule":      convert_rule,
                "is_key":            col_name in keys,
                "key_order":         keys.index(col_name) + 1 if col_name in keys else None,
                "is_toast":          col_name in toast_cols,
                "column_order":      col_order,
                "is_active":         bool(col_cfg.get("active", True)),
                "description":       None,
                "convert_arg":       None if col_cfg.get("arg") is None else __import__("json").dumps(col_cfg["arg"], ensure_ascii=True, separators=(",", ":")),
            })
    return rows


def validate_rows(configs: list, rows: list) -> None:
    """Dừng ngay nếu cấu hình sai, trước khi ghi."""
    errors = []

    seen = set()
    for r in rows:
        k = (r["trg_schema"], r["trg_tbl"], r["trg_column"])
        if k in seen:
            errors.append(f"Trùng cột: {k}")
        seen.add(k)

    for cfg in configs:
        tbl = cfg["target_table"]
        keys = cfg["key"] if isinstance(cfg["key"], list) else [cfg["key"]]
        missing_keys = [k for k in keys if k not in cfg["columns"]]
        missing_toast = [c for c in cfg.get("toast_columns", []) if c not in cfg["columns"]]
        if missing_keys:
            errors.append(f"{tbl}: khoá không có trong columns {missing_keys}")
        if missing_toast:
            errors.append(f"{tbl}: toast_columns không có trong columns {missing_toast}")

    # [THÊM 03/10 - 3P] tên convert_rule
    errors += [f"{r['trg_tbl']}.{r['trg_column']}: convert_rule không hỗ trợ {r['convert_rule']}"
               for r in rows if r["convert_rule"] not in KNOWN_RULES]
    # [THÊM 04/10 - 3P] cột khoá không được tắt
    errors += [f"{r['trg_tbl']}.{r['trg_column']}: cột khoá không được tắt (active False)"
               for r in rows if r["is_key"] and not r["is_active"]]
    time_rules = {"EPOCH_S_TS", "EPOCH_MS_TS", "EPOCH_US_TS", "ISO_UTC_TS"}
    errors += [f"{r['trg_tbl']}.{r['trg_column']}: {r['convert_rule']} chỉ dùng cho timestamp, đang là {r['data_type']}"
               for r in rows if r["convert_rule"] in time_rules and r["data_type"] != "timestamp"]
    errors += [f"{r['trg_tbl']}.{r['trg_column']}: tên cột kỹ thuật không được nằm trong registry"
               for r in rows if r["trg_column"] in {"created_at", "updated_at", "_ingested_at"}]

    arg_rules = {"ARRAY_PICK", "ARRAY_AT", "ENUM_MAP", "ARRAY_PROJECT", "JSON_PICK"}
    errors += [f"{r['trg_tbl']}.{r['trg_column']}: rule có đối số thì convert_arg không được rỗng"
               for r in rows if r["convert_rule"] in arg_rules and not r.get("convert_arg")]
    errors += [f"{r['trg_tbl']}.{r['trg_column']}: convert_arg chỉ dùng cho rule có đối số"
               for r in rows if r["convert_rule"] not in arg_rules and r.get("convert_arg")]
    errors += [f"{r['trg_tbl']}.{r['trg_column']}: json_path nhiều phần không hợp lệ"
               for r in rows if "," in (r["json_path"] or "") and r["convert_rule"] not in {"HASH_MD5_UUID", "HASH_SHA256", "HASH_SHA256_PIPE", "COALESCE", "JSON_PICK"}]
    errors += [f"{r['trg_tbl']}.{r['trg_column']}: json_path không được NULL"
               for r in rows if r["json_path"] is None]
    errors += [f"{r['trg_tbl']}.{r['trg_column']}: ARRAY_AT chỉ dùng cho string"
               for r in rows if r["convert_rule"] == "ARRAY_AT" and r["data_type"] != "string"]
    if errors:
        raise ValueError("Cấu hình TABLE_CONFIGS sai:\n  - " + "\n  - ".join(errors))


# [SỬA 03/10 - 3P] gói thành hàm (logic như cũ) để dùng cho cả nguồn 3rd-party
# END_REGISTRY_PURE

from pyspark.sql.types import (
    BooleanType, IntegerType, StringType, StructField, StructType,
)

REGISTRY_SCHEMA = StructType([
    StructField("src_schema",        StringType(),  False),
    StructField("src_tbl",           StringType(),  False),
    StructField("src_object_schema", StringType(),  True),
    StructField("src_object",        StringType(),  False),
    StructField("trg_schema",        StringType(),  False),
    StructField("trg_tbl",           StringType(),  False),
    StructField("trg_column",        StringType(),  False),
    StructField("json_path",         StringType(),  True),
    StructField("data_type",         StringType(),  False),
    StructField("convert_rule",      StringType(),  False),
    StructField("is_key",            BooleanType(), False),
    StructField("key_order",         IntegerType(), True),
    StructField("is_toast",          BooleanType(), False),
    StructField("column_order",      IntegerType(), False),
    StructField("is_active",         BooleanType(), False),
    StructField("description",       StringType(),  True),
    StructField("convert_arg",       StringType(),  True),
])


In [ ]:
# BEGIN_INIT_CTRL_SEED
_FL00, _ENR = "PL_VV_TRANSFORM_BRONZE_TO_SILVER_FL_00", "PL_VV_TRANS_SLV_TO_ENR_15P"
_B, _S, _E = "lh_vv_bronze.dbo", "lh_vv_silver.dbo", "lh_vv_gold.enr"
# (pl_name, src_schema, src_tbl, trg_schema, trg_tbl, is_active, priority, load_mode, align_path, dedup_order, nb_name)
PIPELINE_CONFIG_ROWS = [
    # [SỬA 21/10] 20261021-move-poi-nodes-enr-15p-cleanup: bỏ 8 cạnh (slv_pn_poi, gld_enr_poi_coordinate, 3 cạnh info cũ); 9 cạnh node silver thuộc _ENR.
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_products', 1, 2, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_product_translations', 1, 2, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_product_attributes', 1, 2, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_product_attribute_translations', 1, 2, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_product_attribute_values', 1, 2, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_product_attribute_value_translations', 1, 2, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_product_variants', 1, 2, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_product_variant_translations', 1, 2, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_product_business_service_relation', 1, 2, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_product_posts', 1, 2, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_product_post_translations', 1, 2, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_partners', 1, 1, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_business_services', 1, 1, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_business_locations', 1, 1, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_hotel_room_types', 1, 1, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_hotel_rate_plans', 1, 1, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_business_service_i18ns', 1, 1, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_business_service_poi_link', 1, 1, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_orders', 1, 1, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_order_refs', 1, 1, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_order_item_tickets', 1, 1, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_order_item_hotels', 1, 1, 'CDC', None, None, None),
    (_FL00, _B, 'partner_raw_data', _S, 'slv_pn_order_item_flights', 1, 1, 'CDC', None, None, None),
    (_FL00, _B, 'brz_3rd_crawler_poi_stream', _S, 'slv_3p_poi', 1, 1, 'DOC', None, None, None),
    (_FL00, _B, 'brz_3rd_crawler_poi_stream', _S, 'slv_3p_poi_address', 1, 1, 'DOC', None, None, None),
    (_FL00, _B, 'brz_3rd_crawler_poi_stream', _S, 'slv_3p_poi_contact', 1, 1, 'DOC', None, None, None),
    (_FL00, _B, 'brz_3rd_crawler_poi_stream', _S, 'slv_3p_poi_opening_hours', 1, 1, 'DOC', None, None, None),
    (_FL00, _B, 'brz_3rd_crawler_poi_stream', _S, 'slv_3p_poi_rating', 1, 1, 'DOC', None, None, None),
    (_FL00, _B, 'brz_3rd_crawler_poi_stream', _S, 'slv_3p_poi_amenity', 1, 1, 'DOC', None, None, None),
    (_FL00, _B, 'brz_3rd_crawler_poi_stream', _S, 'slv_3p_poi_review', 1, 1, 'DOC_ARRAY', None, None, None),
    (_FL00, _B, 'brz_3rd_crawler_poi_stream', _S, 'slv_3p_poi_media', 1, 1, 'DOC_ARRAY', None, 'display_order ASC', None),
    (_FL00, _B, 'brz_3rd_crawler_poi_stream', _S, 'slv_3p_poi_price', 1, 1, 'DOC', None, None, None),
    (_ENR, _S, 'slv_3p_poi_review', _E, 'gld_enr_poi_review', 1, 1, 'APPEND', None, None, 'NB_ENR_POI_REVIEW'),
    (_ENR, _S, 'slv_3p_poi_rating', _E, 'gld_enr_poi_info', 1, 1, 'APPEND', None, None, 'NB_ENR_POI_INFO'),
    (_ENR, _S, 'slv_3p_poi_price', _E, 'gld_enr_poi_info', 1, 1, 'APPEND', None, None, 'NB_ENR_POI_INFO'),
    (_ENR, _S, 'slv_3p_poi_opening_hours', _E, 'gld_enr_poi_info', 1, 1, 'APPEND', None, None, 'NB_ENR_POI_INFO'),
    (_ENR, _S, 'slv_3p_poi_review', _E, 'gld_enr_poi_info', 1, 1, 'APPEND', None, None, 'NB_ENR_POI_INFO'),
    (_ENR, _S, 'slv_poi_entity_non_enrich', _E, 'gld_enr_poi_info', 1, 1, 'APPEND', None, None, 'NB_ENR_POI_INFO'),
    (_ENR, _S, 'slv_poi_amenity', _E, 'gld_enr_poi_info', 1, 1, 'APPEND', None, None, 'NB_ENR_POI_INFO'),
    (_ENR, _S, 'slv_pn_business_service_i18ns', _E, 'gld_enr_poi_info', 1, 1, 'APPEND', None, None, 'NB_ENR_POI_INFO'),
    # [SỬA 22/10] 20261022-enr-landing-and-gld-flags: cạnh landing để cờ làm mới.
    (_ENR, _S, 'slv_enr_poi_info_multi_lang', _E, 'gld_enr_poi_info', 1, 1, 'APPEND', None, None, 'NB_ENR_POI_INFO'),
    (_ENR, _S, 'slv_enr_poi_destination', _E, 'gld_enr_poi_info', 1, 1, 'APPEND', None, None, 'NB_ENR_POI_INFO'),
    (_ENR, _S, 'slv_enr_poi_review_multi_lang', _E, 'gld_enr_poi_review', 1, 1, 'APPEND', None, None, 'NB_ENR_POI_REVIEW'),
    (_ENR, _S, 'slv_3p_poi', _S, 'slv_poi_entity_non_enrich', 1, 1, 'RECOMPUTE', None, None, None),
    (_ENR, _S, 'slv_3p_poi_address', _S, 'slv_poi_entity_non_enrich', 1, 1, 'RECOMPUTE', None, None, None),
    (_ENR, _S, 'ref_dictionary_place_type', _S, 'slv_poi_entity_non_enrich', 1, 1, 'RECOMPUTE', None, None, None),
    (_ENR, _S, 'slv_pn_business_services', _S, 'slv_poi_entity_non_enrich', 1, 1, 'RECOMPUTE', None, None, None),
    (_ENR, _S, 'slv_pn_business_service_i18ns', _S, 'slv_poi_entity_non_enrich', 1, 1, 'RECOMPUTE', None, None, None),
    (_ENR, _S, 'slv_3p_poi_contact', _S, 'slv_poi_entity_non_enrich', 1, 1, 'RECOMPUTE', None, None, None),
    (_ENR, _S, 'slv_3p_poi_amenity', _S, 'slv_poi_amenity', 1, 1, 'RECOMPUTE', None, None, None),
    (_ENR, _S, 'slv_poi_entity_non_enrich', _S, 'slv_poi_amenity', 1, 1, 'RECOMPUTE', None, None, None),
    (_ENR, _S, 'ref_dictionary_amenities', _S, 'slv_poi_amenity', 1, 1, 'RECOMPUTE', None, None, None),
]
# (watermark_id, src_schema, src_tbl, watermark_column, status, flow_name, watermark_sequence)
WATERMARK_ROWS = [
    ("wm_transform_partner_raw_data", _B, "partner_raw_data", "EventProcessedUtcTime", "INITIALIZED",
     "NB_EXTRACT_PARTNER_CDC_BRZ_TO_SLV", -1),
    ("wm_transform_brz_3rd_crawler_poi_stream", _B, "brz_3rd_crawler_poi_stream", "crawled_at", "INITIALIZED",
     "NB_EXTRACT_3RD_PARTY_CDC_BRZ_TO_SLV", -1),
]
# END_INIT_CTRL_SEED


In [ ]:
# BEGIN_PLACE_TYPE_SEED
import re

SEED_SECTORS = (
    "accommodation",
    "f_n_b",
    "spa_n_wellness",
    "shopping_n_retail",
    "attraction",
    "vehicle_rental",
    "other",
)
SEED_SKIP = (
    "Partner Portal bổ sung",
    "Partner Portal Bổ sung",
    "Fallback",
    "Không có",
)
SEED_ROWS = [
    ('partner', 'Hotel', 'accommodation', 'Hotel', 'Khách sạn', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'hotel', 'accommodation', 'Hotel', 'Khách sạn', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'extended_stay_hotel', 'accommodation', 'Extended Stay Hotel', 'Lưu trú dài hạn', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'inn', 'accommodation', 'Inn', 'Khách sạn nhỏ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Resort', 'accommodation', 'Resort', 'Khu nghỉ dưỡng', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'resort_hotel', 'accommodation', 'Resort', 'Khu nghỉ dưỡng', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('partner', 'Hostel', 'accommodation', 'Hostel', 'Hostel', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'hostel', 'accommodation', 'Hostel', 'Hostel', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('partner', 'Motel', 'accommodation', 'Motel', 'Motel', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'motel', 'accommodation', 'Motel', 'Motel', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('partner', 'Homestay', 'accommodation', 'Homestay', 'Homestay', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'guest_house', 'accommodation', 'Homestay', 'Homestay', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'private_guest_room', 'accommodation', 'Homestay', 'Homestay', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'farmstay', 'accommodation', 'Farmstay', 'Farmstay', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Campground/Glamping', 'accommodation', 'Campground', 'Khu cắm trại', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'campground', 'accommodation', 'Campground', 'Khu cắm trại', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'camping_cabin', 'accommodation', 'Camping Cabin', 'Nhà gỗ cắm trại', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bed_and_breakfast', 'accommodation', 'Bed & Breakfast', 'Bed & Breakfast', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'cottage', 'accommodation', 'Cottage', 'Nhà gỗ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'mobile_home_park', 'accommodation', 'Mobile Home Park', 'Khu nhà lưu động', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'lodging', 'accommodation', 'Accommodation', 'Lưu trú', 'Lodging', 1200, 1200, 1200, 'all', 'all_year'),
    ('google', 'budget_japanese_inn', 'accommodation', 'Inn', 'Khách sạn nhỏ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'japanese_inn', 'accommodation', 'Inn', 'Khách sạn nhỏ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Vietnamese', 'f_n_b', 'Vietnamese restaurant', 'Nhà hàng Việt', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'vietnamese_restaurant', 'f_n_b', 'Vietnamese restaurant', 'Nhà hàng Việt', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Chinese', 'f_n_b', 'Chinese restaurant', 'Nhà hàng Trung', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'chinese_restaurant', 'f_n_b', 'Chinese restaurant', 'Nhà hàng Trung', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Japanese', 'f_n_b', 'Japanese restaurant', 'Nhà hàng Nhật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'japanese_restaurant', 'f_n_b', 'Japanese restaurant', 'Nhà hàng Nhật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Korean', 'f_n_b', 'Korean restaurant', 'Nhà hàng Hàn', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'korean_restaurant', 'f_n_b', 'Korean restaurant', 'Nhà hàng Hàn', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Thai', 'f_n_b', 'Thai restaurant', 'Nhà hàng Thái', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'thai_restaurant', 'f_n_b', 'Thai restaurant', 'Nhà hàng Thái', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'French', 'f_n_b', 'French restaurant', 'Nhà hàng Pháp', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'french_restaurant', 'f_n_b', 'French restaurant', 'Nhà hàng Pháp', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Italian', 'f_n_b', 'Italian restaurant', 'Nhà hàng Ý', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'italian_restaurant', 'f_n_b', 'Italian restaurant', 'Nhà hàng Ý', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'pizza_restaurant', 'f_n_b', 'Italian restaurant', 'Nhà hàng Ý', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'American', 'f_n_b', 'American restaurant', 'Nhà hàng Mỹ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'american_restaurant', 'f_n_b', 'American restaurant', 'Nhà hàng Mỹ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Western', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'european_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'british_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'german_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'spanish_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'greek_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'steak_house', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'portuguese_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'austrian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bavarian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'belgian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'danish_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'dutch_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'scandinavian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'swiss_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'polish_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'romanian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'russian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'ukrainian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'croatian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'czech_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hungarian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'australian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'californian_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'tex_mex_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fish_and_chips_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'basque_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'irish_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'soul_food_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'southwestern_us_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'eastern_european_restaurant', 'f_n_b', 'Western restaurant', 'Nhà hàng Châu Âu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'International', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'indian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'north_indian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'south_indian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'indonesian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'malaysian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'filipino_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cambodian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'burmese_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'tibetan_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'taiwanese_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'afghani_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'african_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'ethiopian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'moroccan_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'lebanese_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'israeli_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'persian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'turkish_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'middle_eastern_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'mediterranean_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'mexican_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'latin_american_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'south_american_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'brazilian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'argentinian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'colombian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'chilean_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'peruvian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cuban_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'caribbean_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hawaiian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'asian_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'asian_fusion_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'mongolian_barbecue_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'pakistani_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bangladeshi_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sri_lankan_restaurant', 'f_n_b', 'International', 'Nhà hàng Quốc tế', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Fusion', 'f_n_b', 'Fusion', 'Ẩm thực Fusion', 'Main Meals', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fusion_restaurant', 'f_n_b', 'Fusion', 'Ẩm thực Fusion', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bistro', 'f_n_b', 'Fusion', 'Ẩm thực Fusion', 'Main Meals', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Bakery & Dessert', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Light Snacks / Cafe', 45, 60, 60, 'all', 'all_year'),
    ('google', 'bakery', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Light Snacks / Cafe', 45, 60, 60, 'all', 'all_year'),
    ('google', 'cake_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'pastry_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'dessert_restaurant', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'dessert_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'donut_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'ice_cream_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'confectionery', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'chocolate_shop', 'f_n_b', 'Bakery & Dessert', 'Bánh & Tráng miệng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Bar & Pub/Nightlife', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Shopping & Nightlife', 60, 180, 120, 'all', 'all_year'),
    ('google', 'bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Shopping & Nightlife', 60, 180, 120, 'all', 'all_year'),
    ('google', 'pub', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'night_club', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Shopping & Nightlife', 60, 180, 120, 'all', 'all_year'),
    ('google', 'cocktail_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'wine_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'lounge_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sports_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hookah_bar', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'irish_pub', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'beer_garden', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'karaoke', 'f_n_b', 'Bar & Pub / Nightlife', 'Bar & Pub/ Nightlife', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Fast Food', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Main Meals', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fast_food_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'meal_takeaway', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Main Meals', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sandwich_shop', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'deli', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'snack_bar', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'burrito_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'taco_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'chicken_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'kebab_shop', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'gyro_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'shawarma_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hamburger_restaurant', 'f_n_b', 'Fast Food', 'Đồ ăn nhanh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Vegan Restaurant', 'f_n_b', 'Vegan Restaurant', 'Nhà hàng Chay', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'vegan_restaurant', 'f_n_b', 'Vegan Restaurant', 'Nhà hàng Chay', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'vegetarian_restaurant', 'f_n_b', 'Vegan Restaurant', 'Nhà hàng Chay', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Seafood Restaurant', 'f_n_b', 'Seafood Restaurant', 'Nhà hàng Hải sản', 'Main Meals', 60, 90, 75, 'all', 'all_year'),
    ('google', 'seafood_restaurant', 'f_n_b', 'Seafood Restaurant', 'Nhà hàng Hải sản', 'Main Meals', 60, 90, 75, 'all', 'all_year'),
    ('google', 'oyster_bar_restaurant', 'f_n_b', 'Seafood Restaurant', 'Nhà hàng Hải sản', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Café', 'f_n_b', 'Café', 'Cà phê', 'Light Snacks / Cafe', 45, 60, 60, 'all', 'all_year'),
    ('google', 'coffee_shop', 'f_n_b', 'Café', 'Cà phê', 'Light Snacks / Cafe', 45, 60, 60, 'all', 'all_year'),
    ('google', 'cafe', 'f_n_b', 'Café', 'Cà phê', 'Light Snacks / Cafe', 45, 60, 60, 'all', 'all_year'),
    ('google', 'coffee_roastery', 'f_n_b', 'Café', 'Cà phê', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'coffee_stand', 'f_n_b', 'Café', 'Cà phê', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'tea_house', 'f_n_b', 'Café', 'Cà phê', 'Light Snacks / Cafe', 45, 60, 60, 'all', 'all_year'),
    ('google', 'juice_shop', 'f_n_b', 'Café', 'Cà phê', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hot_pot_restaurant', 'f_n_b', 'Hot Pot', 'Lẩu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'barbecue_restaurant', 'f_n_b', 'Barbecue & Grill', 'Nướng & BBQ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bar_and_grill', 'f_n_b', 'Barbecue & Grill', 'Nướng & BBQ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'korean_barbecue_restaurant', 'f_n_b', 'Barbecue & Grill', 'Nướng & BBQ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'noodle_shop', 'f_n_b', 'Noodle', 'Mì & Món nước', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'ramen_restaurant', 'f_n_b', 'Noodle', 'Mì & Món nước', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'chinese_noodle_restaurant', 'f_n_b', 'Noodle', 'Mì & Món nước', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fine_dining_restaurant', 'f_n_b', 'Fine Dining', 'Fine Dining', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'buffet_restaurant', 'f_n_b', 'Buffet', 'Buffet', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'brunch_restaurant', 'f_n_b', 'Brunch & Breakfast', 'Bữa sáng & Brunch', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'breakfast_restaurant', 'f_n_b', 'Brunch & Breakfast', 'Bữa sáng & Brunch', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'halal_restaurant', 'f_n_b', 'Halal', 'Nhà hàng Halal', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'gastropub', 'f_n_b', 'Gastropub & Brewery', 'Gastropub & Brewery', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'brewpub', 'f_n_b', 'Gastropub & Brewery', 'Gastropub & Brewery', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'brewery', 'f_n_b', 'Gastropub & Brewery', 'Gastropub & Brewery', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'winery', 'f_n_b', 'Gastropub & Brewery', 'Gastropub & Brewery', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'restaurant', 'f_n_b', 'F&B', 'Ẩm thực', 'Main Meals', 60, 90, 75, 'all', 'all_year'),
    ('google', 'food_court', 'f_n_b', 'F&B', 'Ẩm thực', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cafeteria', 'f_n_b', 'F&B', 'Ẩm thực', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'diner', 'f_n_b', 'F&B', 'Ẩm thực', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Day Spa', 'spa_n_wellness', 'Spa', 'Spa', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'spa', 'spa_n_wellness', 'Spa', 'Spa', 'Other', 60, 90, 75, 'spa_wellness', 'all_year'),
    ('google', 'day_spa', 'spa_n_wellness', 'Spa', 'Spa', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'wellness_center', 'spa_n_wellness', 'Spa', 'Spa', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Therapeutic Massage', 'spa_n_wellness', 'Therapeutic Massage', 'Massage trị liệu', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'massage', 'spa_n_wellness', 'Therapeutic Massage', 'Massage trị liệu', 'Other', 60, 90, 75, 'spa_wellness', 'all_year'),
    ('partner', 'Clinic-Aesthetic Spa', 'spa_n_wellness', 'Clinic-Aesthetic Spa', 'Viện thẩm mỹ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'skin_care_clinic', 'spa_n_wellness', 'Clinic-Aesthetic Spa', 'Viện thẩm mỹ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'acupuncture_clinic', 'spa_n_wellness', 'Clinic-Aesthetic Spa', 'Viện thẩm mỹ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'ayurvedic_clinic', 'spa_n_wellness', 'Clinic-Aesthetic Spa', 'Viện thẩm mỹ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Blind Massage', 'spa_n_wellness', 'Blind Massage', 'Massage người khiếm thị', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Onsen-Jjimjilbang', 'spa_n_wellness', 'Onsen-Jjimjilbang', 'Tắm Onsen – Xông hơi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hot_spring', 'spa_n_wellness', 'Onsen-Jjimjilbang', 'Tắm Onsen – Xông hơi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sauna', 'spa_n_wellness', 'Onsen-Jjimjilbang', 'Tắm Onsen – Xông hơi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Hair Salon', 'spa_n_wellness', 'Hair Salon', 'Salon', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hair_salon', 'spa_n_wellness', 'Hair Salon', 'Salon', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hair_care', 'spa_n_wellness', 'Hair Salon', 'Salon', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Nail-Lash Salon', 'spa_n_wellness', 'Nail-Lash Salon', 'Tiệm Nail – Mi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'nail_salon', 'spa_n_wellness', 'Nail-Lash Salon', 'Tiệm Nail – Mi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Gym-Fitness', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'gym', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fitness_center', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sports_club', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'recreation_center', 'spa_n_wellness', 'Gym-Fitness', 'Phòng Gym – Thể hình', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Yoga-Pilates Studio', 'spa_n_wellness', 'Yoga-Pilates Studio', 'Phòng tập Yoga – Pilates', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'yoga_studio', 'spa_n_wellness', 'Yoga-Pilates Studio', 'Phòng tập Yoga – Pilates', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'pilates_studio', 'spa_n_wellness', 'Yoga-Pilates Studio', 'Phòng tập Yoga – Pilates', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Meditation Center', 'spa_n_wellness', 'Meditation Center', 'Trung tâm Thiền', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'meditation_center', 'spa_n_wellness', 'Meditation Center', 'Trung tâm Thiền', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'beauty_salon', 'spa_n_wellness', 'Beauty Salon', 'Spa & Làm đẹp', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hair_removal_service', 'spa_n_wellness', 'Hair Removal Service', 'Dịch vụ triệt lông', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'tanning_studio', 'spa_n_wellness', 'Tanning Studio', 'Dịch vụ nhuộm da', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'barber_shop', 'spa_n_wellness', 'Barbershop', 'Barbershop', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'body_piercing_shop', 'spa_n_wellness', 'Tattoo & Piercing', 'Xăm hình & Xỏ khuyên', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'tattoo_parlor', 'spa_n_wellness', 'Tattoo & Piercing', 'Xăm hình & Xỏ khuyên', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'swimming_pool', 'spa_n_wellness', 'Swimming Pool', 'Bể bơi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Local Specialty', 'shopping_n_retail', 'Local Specialty', 'Đặc sản địa phương', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Souvenirs & Handicrafts', 'shopping_n_retail', 'Souvenirs & Handicrafts', 'Quà lưu niệm & Thủ công mỹ nghệ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'gift_shop', 'shopping_n_retail', 'Souvenirs & Handicrafts', 'Quà lưu niệm & Thủ công mỹ nghệ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'antique_shop', 'shopping_n_retail', 'Souvenirs & Handicrafts', 'Quà lưu niệm & Thủ công mỹ nghệ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Fashion & Accessories', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'clothing_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'shoe_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fashion_accessories_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hat_shop', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sunglasses_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'swimwear_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'leather_goods_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bag_store', 'shopping_n_retail', 'Fashion & Accessories', 'Thời trang & Phụ kiện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Jewelry & Gems', 'shopping_n_retail', 'Jewelry & Gems', 'Trang sức & Đá quý', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'jewelry_store', 'shopping_n_retail', 'Jewelry & Gems', 'Trang sức & Đá quý', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Cosmetics & Beauty', 'shopping_n_retail', 'Cosmetics & Beauty', 'Mỹ phẩm & Làm đẹp', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cosmetics_store', 'shopping_n_retail', 'Cosmetics & Beauty', 'Mỹ phẩm & Làm đẹp', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'perfume_store', 'shopping_n_retail', 'Cosmetics & Beauty', 'Mỹ phẩm & Làm đẹp', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'grocery_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'food_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'liquor_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'home_goods_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'furniture_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'home_improvement_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hardware_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'book_store', 'shopping_n_retail', 'Books & Stationery', 'Sách – Văn phòng phẩm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'stationery_store', 'shopping_n_retail', 'Books & Stationery', 'Sách – Văn phòng phẩm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'electronics_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cell_phone_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'shopping_mall', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị', 'Shopping & Nightlife', 60, 180, 120, 'shopping_nightlife', 'all_year'),
    ('google', 'supermarket', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'department_store', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'discount_store', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'wholesaler', 'shopping_n_retail', 'Mall - Supermarket', 'Trung tâm thương mại/Siêu thị', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Convenience Store', 'shopping_n_retail', 'Convenience Store', 'Cửa hàng tiện lợi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'convenience_store', 'shopping_n_retail', 'Convenience Store', 'Cửa hàng tiện lợi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Local/Night Market', 'shopping_n_retail', 'Local/ Night Market', 'Chợ truyền thống / Chợ đêm', 'Shopping & Nightlife', 60, 180, 120, 'all', 'all_year'),
    ('google', 'market', 'shopping_n_retail', 'Local/ Night Market', 'Chợ truyền thống / Chợ đêm', 'Shopping & Nightlife', 90, 150, 120, 'shopping_nightlife', 'all_year'),
    ('partner', 'Sporting Goods', 'shopping_n_retail', 'Sporting Goods', 'Dụng cụ thể thao', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sporting_goods_store', 'shopping_n_retail', 'Sporting Goods', 'Dụng cụ thể thao', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bicycle_store', 'shopping_n_retail', 'Sporting Goods', 'Dụng cụ thể thao', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'toy_store', 'shopping_n_retail', 'Toy Store', 'Cửa hàng đồ chơi', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'baby_goods_store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'luggage_store', 'shopping_n_retail', 'Luggage Store', 'Vali & Phụ kiện du lịch', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'second_hand_shop', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'florist', 'shopping_n_retail', 'Florist', 'Cửa hàng hoa', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'store', 'shopping_n_retail', 'Shopping & Retail', 'Mua sắm', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Home & Decor', 'shopping_n_retail', 'Home & Decor', 'Đồ gia dụng - Gốm sứ', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Theme Park', 'attraction', 'Theme Park', 'Công viên giải trí', 'Heavy Entertainment', 180, 360, 270, 'all', 'all_year'),
    ('google', 'amusement_park', 'attraction', 'Theme Park', 'Công viên giải trí', 'Heavy Entertainment', 180, 360, 270, 'outdoor_entertainment', 'Month 11–4'),
    ('google', 'amusement_center', 'attraction', 'Theme Park', 'Công viên giải trí', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fairground', 'attraction', 'Theme Park', 'Công viên giải trí', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Water Park', 'attraction', 'Water Park', 'Công viên nước', 'Heavy Entertainment', 180, 360, 270, 'all', 'all_year'),
    ('google', 'water_park', 'attraction', 'Water Park', 'Công viên nước', 'Heavy Entertainment', 180, 360, 270, 'outdoor_entertainment', 'Month 11–4'),
    ('partner', 'Museum', 'attraction', 'Museum', 'Bảo tàng', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'all', 'all_year'),
    ('google', 'museum', 'attraction', 'Museum', 'Bảo tàng', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'indoor_culture', 'all_year'),
    ('google', 'planetarium', 'attraction', 'Museum', 'Bảo tàng', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Historical Site', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'all', 'all_year'),
    ('google', 'historical_landmark', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'all', 'all_year'),
    ('google', 'archaeological_site', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'ruins', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'castle', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'palace', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'monument', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'sculpture', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'lighthouse', 'attraction', 'Historical Site', 'Di tích lịch sử', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Zoo-Aquarium', 'attraction', 'Zoo-Aquarium', 'Vườn thú – Thủy cung', 'Heavy Entertainment', 180, 360, 270, 'all', 'all_year'),
    ('google', 'zoo', 'attraction', 'Zoo-Aquarium', 'Vườn thú – Thủy cung', 'Heavy Entertainment', 180, 360, 270, 'all', 'all_year'),
    ('google', 'aquarium', 'attraction', 'Zoo-Aquarium', 'Vườn thú – Thủy cung', 'Heavy Entertainment', 180, 360, 270, 'all', 'all_year'),
    ('google', 'wildlife_park', 'attraction', 'Zoo-Aquarium', 'Vườn thú – Thủy cung', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Nature Reserve', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên', 'Heavy Entertainment', 180, 360, 270, 'all', 'all_year'),
    ('google', 'nature_reserve', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'national_park', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên', 'Heavy Entertainment', 180, 360, 270, 'forest_hiking', 'Month 11–3'),
    ('google', 'hiking_area', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên', 'Heavy Entertainment', 180, 360, 270, 'forest_hiking', 'Month 11–3'),
    ('google', 'wildlife_refuge', 'attraction', 'Nature Reserve', 'Khu bảo tồn thiên nhiên', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Architecture-Landmark', 'attraction', 'Architecture-Landmark', 'Công trình kiến trúc', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cultural_landmark', 'attraction', 'Architecture-Landmark', 'Công trình kiến trúc', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Art Show-Performance', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'concert_hall', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'performing_arts_theater', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'opera_house', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'comedy_club', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'amphitheatre', 'attraction', 'Art Show-Performance', 'Show diễn nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Traditional Village', 'attraction', 'Traditional Village', 'Làng nghề', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Religious Site', 'attraction', 'Religious Site', 'Điểm đến tâm linh', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'all', 'all_year'),
    ('google', 'church', 'attraction', 'Religious Site', 'Điểm đến tâm linh', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'indoor_culture', 'all_year'),
    ('google', 'hindu_temple', 'attraction', 'Religious Site', 'Điểm đến tâm linh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'mosque', 'attraction', 'Religious Site', 'Điểm đến tâm linh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'synagogue', 'attraction', 'Religious Site', 'Điểm đến tâm linh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'buddhist_temple', 'attraction', 'Religious Site', 'Điểm đến tâm linh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'place_of_worship', 'attraction', 'Religious Site', 'Điểm đến tâm linh', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'indoor_culture', 'all_year'),
    ('partner', 'Casino', 'attraction', 'Casino', 'Sòng bạc', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'casino', 'attraction', 'Casino', 'Sòng bạc', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Art Gallery', 'attraction', 'Art Gallery', 'Phòng tranh/ Xưởng nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'art_gallery', 'attraction', 'Art Gallery', 'Phòng tranh/ Xưởng nghệ thuật', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cultural_center', 'attraction', 'Cultural Center', 'Trung tâm văn hóa', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'community_center', 'attraction', 'Cultural Center', 'Trung tâm văn hóa', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'escape_room', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'video_arcade', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bowling_alley', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'miniature_golf_course', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'go_kart_track', 'attraction', 'Indoor Entertainment', 'Khu vui chơi trong nhà', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'beach', 'attraction', 'Beach', 'Biển', 'Light Sightseeing / Culture & Nature', 120, 240, 180, 'beach_coast', 'Month 12–4'),
    ('google', 'park', 'attraction', 'Park', 'Công viên', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'all', 'all_year'),
    ('google', 'viewpoint', 'attraction', 'Viewpoint', 'Điểm ngắm cảnh', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'observation_deck', 'attraction', 'Observation Deck', 'Đài quan sát', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'waterfall', 'attraction', 'Waterfall', 'Thác nước', 'Other', 60, 90, 75, 'waterfall_green_season', 'Month 6–10'),
    ('google', 'cave', 'attraction', 'Cave', 'Hang động', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'botanical_garden', 'attraction', 'Botanical Garden', 'Vườn bách thảo', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'stadium', 'attraction', 'Stadium', 'Sân vận động', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Motorbike Rental', 'vehicle_rental', 'Motorbike Rental', 'Cho thuê xe máy', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'scooter_rental', 'vehicle_rental', 'Motorbike Rental', 'Cho thuê xe máy', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Self-drive Car Rental', 'vehicle_rental', 'Self-drive Car Rental', 'Cho thuê ô tô tự lái', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'car_rental', 'vehicle_rental', 'Self-drive Car Rental', 'Cho thuê ô tô tự lái', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Car with Driver', 'vehicle_rental', None, None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Bicycle-EV Rental', 'vehicle_rental', 'Bicycle-EV Rental', 'Cho thuê xe đạp – Xe điện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bicycle_rental', 'vehicle_rental', 'Bicycle-EV Rental', 'Cho thuê xe đạp – Xe điện', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('partner', 'Campervan Rental', 'vehicle_rental', None, 'Cho thuê xe nhà di động', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'boat_rental', 'vehicle_rental', 'Boat Rental', 'Cho thuê thuyền', 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'airport', 'other', 'airport', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'international_airport', 'other', 'airport', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'bus_station', 'other', 'bus_station', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'train_station', 'other', 'train_station', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'subway_station', 'other', 'metro_station', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'light_rail_station', 'other', 'metro_station', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'ferry_terminal', 'other', 'ferry_terminal', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'transit_station', 'other', 'transit_station', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'natural_feature', 'other', 'natural_feature', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'town_square', 'other', 'town_square', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'city_hall', 'other', 'government_offices', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'local_government_office', 'other', 'government_offices', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'embassy', 'other', 'embassy', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'police', 'other', 'police_and_fire_station', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'fire_station', 'other', 'police_and_fire_station', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'post_office', 'other', 'post_office', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'library', 'other', 'public_library', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'hospital', 'other', 'hospital', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'drugstore', 'other', 'drugstore', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'university', 'other', 'academic_building', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'school', 'other', 'academic_building', None, 'Other', 60, 90, 75, 'all', 'all_year'),
    ('google', 'cable_car', 'other', 'other', 'other', 'Other', 180, 360, 270, 'all', 'all_year'),
    ('google', 'safari', 'other', 'other', 'other', 'Other', 180, 360, 270, 'all', 'all_year'),
    ('google', 'theme', 'other', 'other', 'other', 'Other', 180, 360, 270, 'all', 'all_year'),
    ('google', 'vinwonders', 'other', 'other', 'other', 'Other', 180, 360, 270, 'all', 'all_year'),
    ('google', 'night_market', 'other', 'other', 'other', 'Shopping & Nightlife', 90, 150, 120, 'shopping_nightlife', 'all_year'),
    ('google', 'pagoda', 'other', 'other', 'other', 'Light Sightseeing / Culture & Nature', 60, 120, 90, 'indoor_culture', 'all_year'),
    ('google', 'temple', 'other', 'other', 'other', 'Other', 60, 120, 90, 'indoor_culture', 'all_year'),
    ('google', 'prison', 'other', 'other', 'other', 'Other', 60, 120, 90, 'indoor_culture', 'all_year'),
    ('google', 'snorkel', 'other', 'other', 'other', 'Other', 120, 240, 180, 'snorkeling_diving', 'Month 12–5'),
    ('google', 'island', 'other', 'other', 'other', 'Other', 120, 240, 180, 'island_boat', 'Month 11–4'),
    ('google', 'boat', 'other', 'other', 'other', 'Other', 120, 240, 180, 'island_boat', 'Month 11–4'),
    ('google', 'snorkeling', 'other', 'other', 'other', 'Other', 60, 90, 75, 'snorkeling_diving', 'Month 12–5'),
    ('google', 'diving', 'other', 'other', 'other', 'Other', 60, 90, 75, 'snorkeling_diving', 'Month 12–5'),
    ('google', 'scuba', 'other', 'other', 'other', 'Other', 60, 90, 75, 'snorkeling_diving', 'Month 12–5'),
    ('google', 'canoe', 'other', 'other', 'other', 'Other', 60, 90, 75, 'island_boat', 'Month 11–4'),
    ('google', 'ferry', 'other', 'other', 'other', 'Other', 60, 90, 75, 'island_boat', 'Month 11–4'),
    ('google', 'sailing', 'other', 'other', 'other', 'Other', 60, 90, 75, 'island_boat', 'Month 11–4'),
    ('google', 'sunset', 'other', 'other', 'other', 'Other', 60, 90, 75, 'beach_coast', 'Month 12–4'),
    ('google', 'coast', 'other', 'other', 'other', 'Other', 60, 90, 75, 'beach_coast', 'Month 12–4'),
    ('google', 'coastal', 'other', 'other', 'other', 'Other', 60, 90, 75, 'beach_coast', 'Month 12–4'),
    ('google', 'stream', 'other', 'other', 'other', 'Other', 60, 90, 75, 'waterfall_green_season', 'Month 6–10'),
    ('google', 'suối', 'other', 'other', 'other', 'Other', 60, 90, 75, 'waterfall_green_season', 'Month 6–10'),
    ('google', 'thác', 'other', 'other', 'other', 'Other', 60, 90, 75, 'waterfall_green_season', 'Month 6–10'),
    ('google', 'forest', 'other', 'other', 'other', 'Other', 60, 90, 75, 'forest_hiking', 'Month 11–3'),
    ('google', 'hiking', 'other', 'other', 'other', 'Other', 60, 90, 75, 'forest_hiking', 'Month 11–3'),
    ('google', 'trekking', 'other', 'other', 'other', 'Other', 60, 90, 75, 'forest_hiking', 'Month 11–3'),
    ('google', 'trail', 'other', 'other', 'other', 'Other', 60, 90, 75, 'forest_hiking', 'Month 11–3'),
    ('google', 'shopping', 'other', 'other', 'other', 'Other', 60, 90, 75, 'shopping_nightlife', 'all_year'),
    ('google', 'mall', 'other', 'other', 'other', 'Other', 60, 90, 75, 'shopping_nightlife', 'all_year'),
    ('google', 'wellness', 'other', 'other', 'other', 'Other', 60, 90, 75, 'spa_wellness', 'all_year'),
    ('google', 'theme_park', 'other', 'other', 'other', 'Heavy Entertainment', 180, 360, 270, 'outdoor_entertainment', 'Month 11–4'),
]
CATEGORY_OTHER = "Other"
# Nhãn mục 1.8. Chỉ để kiểm tra; giá trị ghi sẵn trong từng dòng SEED_ROWS.
CATEGORY_GROUP_VALUES = (
    "Lodging", "Main Meals", "Light Snacks / Cafe", "Heavy Entertainment",
    "Light Sightseeing / Culture & Nature", "Shopping & Nightlife", CATEGORY_OTHER,
)
# Mục 1.9.2: candidate_best_time_code -> canonical_candidate. Chỉ để kiểm tra.
BEST_TIME_CANONICAL = {
    "snorkeling_diving": "Month 12–5", "island_boat": "Month 11–4", "beach_coast": "Month 12–4",
    "waterfall_green_season": "Month 6–10", "forest_hiking": "Month 11–3",
    "outdoor_entertainment": "Month 11–4", "indoor_culture": "all_year",
    "shopping_nightlife": "all_year", "spa_wellness": "all_year", "all": "all_year",
}
TOKEN_KEEP = ("suối", "thác")  # D-04: giữ đúng chữ


def validate_seed(rows):
    """Khóa trùng, source, sector, placeholder, token google, category_group, ước tính 1.9."""
    seen = set()
    token_re = re.compile(r"^[a-z0-9_]+$")
    allowed = set(SEED_SECTORS)
    skip = set(SEED_SKIP)
    for row in rows:
        if len(row) != 11:
            raise ValueError(f"{row[:2]}: {len(row)} trường, cần 11")
        source, place_type, sector, _cat, _cat_vi, group, low, high, avg, code, canonical = row
        key = (source, place_type)
        if key in seen:
            raise ValueError(f"duplicate key {key}")
        seen.add(key)
        if source not in {"google", "partner"}:
            raise ValueError(f"source {source!r}")
        if sector not in allowed:
            raise ValueError(f"sector {sector!r}")
        if place_type in skip:
            raise ValueError(f"place_type {place_type!r}")
        if source == "google" and place_type not in TOKEN_KEEP and token_re.fullmatch(place_type) is None:
            raise ValueError(f"bad token {place_type!r}")
        if group not in CATEGORY_GROUP_VALUES:
            raise ValueError(f"{key}: category_group {group!r}")
        if not all(type(v) is int for v in (low, high, avg)) or not low <= avg <= high:
            raise ValueError(f"{key}: range {low}-{high} avg {avg}")
        if BEST_TIME_CANONICAL.get(code) != canonical:
            raise ValueError(f"{key}: best time {code!r} -> {canonical!r}")
# END_PLACE_TYPE_SEED

validate_seed(SEED_ROWS)
PLACE_TYPE_ROWS = list(SEED_ROWS)


In [ ]:
# BEGIN_AMENITY_SEED
import re

AMENITY_FLAG_RE = re.compile(r"([A-Za-z][A-Za-z0-9]*(?:\.[A-Za-z][A-Za-z0-9]*)?) = true")
AMENITY_TYPES_RE = re.compile(r"types\[\] = (.+)")
# Khoá 4 cột trùng có sẵn từ mapping, dạng giữ nguyên, không phải key khớp.
# Câu types[] có/thuộc -> json_type. place_type google trong ref_dictionary_place_type.
# Day Spa không có category đúng tên: dùng category Spa.
# electric_vehicle_charging_station không có trong từ điển: giữ token.
AMENITY_JSON_TYPE = {
    'types[] có enum thuộc business_sector F&B': "json_type in ('vietnamese_restaurant', 'chinese_restaurant', 'japanese_restaurant', 'korean_restaurant', 'thai_restaurant', 'french_restaurant', 'italian_restaurant', 'pizza_restaurant', 'american_restaurant', 'european_restaurant', 'british_restaurant', 'german_restaurant', 'spanish_restaurant', 'greek_restaurant', 'steak_house', 'portuguese_restaurant', 'austrian_restaurant', 'bavarian_restaurant', 'belgian_restaurant', 'danish_restaurant', 'dutch_restaurant', 'scandinavian_restaurant', 'swiss_restaurant', 'polish_restaurant', 'romanian_restaurant', 'russian_restaurant', 'ukrainian_restaurant', 'croatian_restaurant', 'czech_restaurant', 'hungarian_restaurant', 'australian_restaurant', 'californian_restaurant', 'tex_mex_restaurant', 'fish_and_chips_restaurant', 'basque_restaurant', 'irish_restaurant', 'soul_food_restaurant', 'southwestern_us_restaurant', 'eastern_european_restaurant', 'indian_restaurant', 'north_indian_restaurant', 'south_indian_restaurant', 'indonesian_restaurant', 'malaysian_restaurant', 'filipino_restaurant', 'cambodian_restaurant', 'burmese_restaurant', 'tibetan_restaurant', 'taiwanese_restaurant', 'afghani_restaurant', 'african_restaurant', 'ethiopian_restaurant', 'moroccan_restaurant', 'lebanese_restaurant', 'israeli_restaurant', 'persian_restaurant', 'turkish_restaurant', 'middle_eastern_restaurant', 'mediterranean_restaurant', 'mexican_restaurant', 'latin_american_restaurant', 'south_american_restaurant', 'brazilian_restaurant', 'argentinian_restaurant', 'colombian_restaurant', 'chilean_restaurant', 'peruvian_restaurant', 'cuban_restaurant', 'caribbean_restaurant', 'hawaiian_restaurant', 'asian_restaurant', 'asian_fusion_restaurant', 'mongolian_barbecue_restaurant', 'pakistani_restaurant', 'bangladeshi_restaurant', 'sri_lankan_restaurant', 'fusion_restaurant', 'bistro', 'bakery', 'cake_shop', 'pastry_shop', 'dessert_restaurant', 'dessert_shop', 'donut_shop', 'ice_cream_shop', 'confectionery', 'chocolate_shop', 'bar', 'pub', 'night_club', 'cocktail_bar', 'wine_bar', 'lounge_bar', 'sports_bar', 'hookah_bar', 'irish_pub', 'beer_garden', 'karaoke', 'fast_food_restaurant', 'meal_takeaway', 'sandwich_shop', 'deli', 'snack_bar', 'burrito_restaurant', 'taco_restaurant', 'chicken_restaurant', 'kebab_shop', 'gyro_restaurant', 'shawarma_restaurant', 'hamburger_restaurant', 'vegan_restaurant', 'vegetarian_restaurant', 'seafood_restaurant', 'oyster_bar_restaurant', 'coffee_shop', 'cafe', 'coffee_roastery', 'coffee_stand', 'tea_house', 'juice_shop', 'hot_pot_restaurant', 'barbecue_restaurant', 'bar_and_grill', 'korean_barbecue_restaurant', 'noodle_shop', 'ramen_restaurant', 'chinese_noodle_restaurant', 'fine_dining_restaurant', 'buffet_restaurant', 'brunch_restaurant', 'breakfast_restaurant', 'halal_restaurant', 'gastropub', 'brewpub', 'brewery', 'winery', 'restaurant', 'food_court', 'cafeteria', 'diner')",
    'types[] có enum thuộc business_category = Bar & Pub/Nightlife servesBeer = true': "json_type in ('bar', 'pub', 'night_club', 'cocktail_bar', 'wine_bar', 'lounge_bar', 'sports_bar', 'hookah_bar', 'irish_pub', 'beer_garden', 'karaoke')",
    'types[] có enum thuộc business_category = Day Spa/Therapeutic Massage/Clinic-Aesthetic Spa/Hair Salon/Nail-Lash Salon/Beauty Salon/Barbershop': "json_type in ('spa', 'day_spa', 'wellness_center', 'massage', 'skin_care_clinic', 'acupuncture_clinic', 'ayurvedic_clinic', 'hair_salon', 'hair_care', 'nail_salon', 'beauty_salon', 'barber_shop')",
    'types[] có enum thuộc business_category = Gym-Fitness/Yoga-Pilates Studio': "json_type in ('gym', 'fitness_center', 'sports_club', 'recreation_center', 'yoga_studio', 'pilates_studio')",
    'types[] có electric_vehicle_charging_station': "json_type = 'electric_vehicle_charging_station'",
    'types[] có enum thuộc business_category = Historical Site': "json_type in ('historical_landmark', 'archaeological_site', 'ruins', 'castle', 'palace', 'monument', 'sculpture', 'lighthouse')",
    'types[] có art_gallery': "json_type = 'art_gallery'",
    'types[] có planetarium': "json_type = 'planetarium'",
    'types[] có business_category = Religious Site': "json_type in ('church', 'hindu_temple', 'mosque', 'synagogue', 'buddhist_temple', 'place_of_worship')",
    'types[] có restaurant, cafe, coffee_shop': "json_type in ('restaurant', 'cafe', 'coffee_shop')",
    'types[] có gift_shop': "json_type = 'gift_shop'",
    'types[] thuộc business_category = Bar & Pub/Nightlife': "json_type in ('bar', 'pub', 'night_club', 'cocktail_bar', 'wine_bar', 'lounge_bar', 'sports_bar', 'hookah_bar', 'irish_pub', 'beer_garden', 'karaoke')",
}


AMENITY_SHARED_RAW = (("google", "attraction", "facilities_and_services", "goodForChildren"),)


def amenity_keys(raw):
    """amenity_raw mapping -> list key. "a.b = true" -> ["a.b"]; "types[] = x, y" -> ["x", "y"]; câu types[] có/thuộc -> json_type."""
    mapped = AMENITY_JSON_TYPE.get(raw)
    if mapped:
        return [mapped]
    flag = AMENITY_FLAG_RE.fullmatch(raw)
    if flag:
        return [flag.group(1)]
    types = AMENITY_TYPES_RE.fullmatch(raw)
    if types:
        return [part.strip() for part in types.group(1).split(",")]
    return [raw]

SEED_SECTORS = ("accommodation", "f_n_b", "attraction", "shopping_n_retail", "other")
SEED_GROUPS = (
    "general_amenities",
    "accessibility",
    "specialty_tags",
    "space_and_services",
    "facilities_and_services",
    "other",
)
SEED_SKIP = ("Bổ sung sau", "Fallback", "Không có field", "Không có field riêng")
SEED_ROWS = [
    ('google', 'accommodation', 'general_amenities', 'parkingOptions.freeStreetParking', 'Free Street Parking', 'Đỗ xe trên phố', 'Parking'),
    ('google', 'accommodation', 'general_amenities', 'parkingOptions.valetParking', 'Valet Parking', 'Dịch vụ Valet', 'Parking'),
    ('google', 'accommodation', 'general_amenities', 'parkingOptions.freeGarageParking', 'Free Garage Parking', 'Bãi xe trong nhà (Miễn phí)', 'Parking'),
    ('google', 'accommodation', 'general_amenities', 'parkingOptions.paidGarageParking', 'Paid Garage Parking', 'Bãi xe trong nhà (Thu phí)', 'Parking'),
    ('google', 'accommodation', 'general_amenities', 'outdoorSeating', 'Outdoor Seating', 'Không gian ngoài trời', 'Food/service'),
    ('google', 'accommodation', 'general_amenities', 'restroom', 'Restroom', 'Nhà vệ sinh', 'Suitability'),
    ('google', 'accommodation', 'general_amenities', 'goodForChildren', 'Good for Children', 'Thân thiện trẻ em', 'Suitability'),
    ('google', 'accommodation', 'general_amenities', 'goodForGroups', 'Good for Groups', 'Phù hợp đoàn nhóm', 'Suitability'),
    ('google', 'accommodation', 'general_amenities', 'liveMusic', 'Live Music', 'Nhạc Live', 'Food/service'),
    ('partner', 'accommodation', 'general_amenities', 'Hồ bơi', 'Swimming Pool', 'Hồ bơi', None),
    ('google', 'accommodation', 'general_amenities', 'swimming_pool', 'Swimming Pool', 'Hồ bơi', 'Derived amenities'),
    ('partner', 'accommodation', 'general_amenities', 'Wi-Fi miễn phí', 'Free Wi-Fi', 'Wi-Fi miễn phí', None),
    ('partner', 'accommodation', 'general_amenities', 'Bãi đậu xe', 'Parking', 'Bãi đậu xe', None),
    ('google', 'accommodation', 'general_amenities', 'parkingOptions.freeParkingLot', 'Parking', 'Bãi đậu xe', 'Parking'),
    ('google', 'accommodation', 'general_amenities', 'parkingOptions.paidParkingLot', 'Parking', 'Bãi đậu xe', 'Parking'),
    ('partner', 'accommodation', 'general_amenities', 'Xe đưa đón sân bay', 'Airport Shuttle', 'Xe đưa đón sân bay', None),
    ('partner', 'accommodation', 'general_amenities', 'Nhà hàng', 'Restaurant', 'Nhà hàng', None),
    ('google', 'accommodation', 'general_amenities', "json_type in ('vietnamese_restaurant', 'chinese_restaurant', 'japanese_restaurant', 'korean_restaurant', 'thai_restaurant', 'french_restaurant', 'italian_restaurant', 'pizza_restaurant', 'american_restaurant', 'european_restaurant', 'british_restaurant', 'german_restaurant', 'spanish_restaurant', 'greek_restaurant', 'steak_house', 'portuguese_restaurant', 'austrian_restaurant', 'bavarian_restaurant', 'belgian_restaurant', 'danish_restaurant', 'dutch_restaurant', 'scandinavian_restaurant', 'swiss_restaurant', 'polish_restaurant', 'romanian_restaurant', 'russian_restaurant', 'ukrainian_restaurant', 'croatian_restaurant', 'czech_restaurant', 'hungarian_restaurant', 'australian_restaurant', 'californian_restaurant', 'tex_mex_restaurant', 'fish_and_chips_restaurant', 'basque_restaurant', 'irish_restaurant', 'soul_food_restaurant', 'southwestern_us_restaurant', 'eastern_european_restaurant', 'indian_restaurant', 'north_indian_restaurant', 'south_indian_restaurant', 'indonesian_restaurant', 'malaysian_restaurant', 'filipino_restaurant', 'cambodian_restaurant', 'burmese_restaurant', 'tibetan_restaurant', 'taiwanese_restaurant', 'afghani_restaurant', 'african_restaurant', 'ethiopian_restaurant', 'moroccan_restaurant', 'lebanese_restaurant', 'israeli_restaurant', 'persian_restaurant', 'turkish_restaurant', 'middle_eastern_restaurant', 'mediterranean_restaurant', 'mexican_restaurant', 'latin_american_restaurant', 'south_american_restaurant', 'brazilian_restaurant', 'argentinian_restaurant', 'colombian_restaurant', 'chilean_restaurant', 'peruvian_restaurant', 'cuban_restaurant', 'caribbean_restaurant', 'hawaiian_restaurant', 'asian_restaurant', 'asian_fusion_restaurant', 'mongolian_barbecue_restaurant', 'pakistani_restaurant', 'bangladeshi_restaurant', 'sri_lankan_restaurant', 'fusion_restaurant', 'bistro', 'bakery', 'cake_shop', 'pastry_shop', 'dessert_restaurant', 'dessert_shop', 'donut_shop', 'ice_cream_shop', 'confectionery', 'chocolate_shop', 'bar', 'pub', 'night_club', 'cocktail_bar', 'wine_bar', 'lounge_bar', 'sports_bar', 'hookah_bar', 'irish_pub', 'beer_garden', 'karaoke', 'fast_food_restaurant', 'meal_takeaway', 'sandwich_shop', 'deli', 'snack_bar', 'burrito_restaurant', 'taco_restaurant', 'chicken_restaurant', 'kebab_shop', 'gyro_restaurant', 'shawarma_restaurant', 'hamburger_restaurant', 'vegan_restaurant', 'vegetarian_restaurant', 'seafood_restaurant', 'oyster_bar_restaurant', 'coffee_shop', 'cafe', 'coffee_roastery', 'coffee_stand', 'tea_house', 'juice_shop', 'hot_pot_restaurant', 'barbecue_restaurant', 'bar_and_grill', 'korean_barbecue_restaurant', 'noodle_shop', 'ramen_restaurant', 'chinese_noodle_restaurant', 'fine_dining_restaurant', 'buffet_restaurant', 'brunch_restaurant', 'breakfast_restaurant', 'halal_restaurant', 'gastropub', 'brewpub', 'brewery', 'winery', 'restaurant', 'food_court', 'cafeteria', 'diner')", 'Restaurant', 'Nhà hàng', 'Derived amenities'),
    ('google', 'accommodation', 'general_amenities', 'serves_meals = true', 'Restaurant', 'Nhà hàng', None),
    ('partner', 'accommodation', 'general_amenities', 'Lễ tân 24h', '24-hour Front Desk', 'Lễ tân 24h', None),
    ('partner', 'accommodation', 'general_amenities', 'Quầy bar', 'Bar', 'Quầy bar', None),
    ('google', 'accommodation', 'general_amenities', "json_type in ('bar', 'pub', 'night_club', 'cocktail_bar', 'wine_bar', 'lounge_bar', 'sports_bar', 'hookah_bar', 'irish_pub', 'beer_garden', 'karaoke')", 'Bar', 'Quầy bar', 'Derived amenities'),
    ('google', 'accommodation', 'general_amenities', 'servesCocktails', 'Bar', 'Quầy bar', 'Food/service'),
    ('partner', 'accommodation', 'general_amenities', 'Spa & Wellness', 'Spa & Wellness', 'Spa & Wellness', None),
    ('google', 'accommodation', 'general_amenities', "json_type in ('spa', 'day_spa', 'wellness_center', 'massage', 'skin_care_clinic', 'acupuncture_clinic', 'ayurvedic_clinic', 'hair_salon', 'hair_care', 'nail_salon', 'beauty_salon', 'barber_shop')", 'Spa & Wellness', 'Spa & Wellness', 'Derived amenities'),
    ('partner', 'accommodation', 'general_amenities', 'Phòng Gym', 'Gym', 'Phòng Gym', None),
    ('google', 'accommodation', 'general_amenities', "json_type in ('gym', 'fitness_center', 'sports_club', 'recreation_center', 'yoga_studio', 'pilates_studio')", 'Gym', 'Phòng Gym', 'Derived amenities'),
    ('partner', 'accommodation', 'general_amenities', 'Xông hơi', 'Sauna', 'Xông hơi', None),
    ('google', 'accommodation', 'general_amenities', 'hot_spring', 'Sauna', 'Xông hơi', 'Derived amenities'),
    ('google', 'accommodation', 'general_amenities', 'sauna', 'Sauna', 'Xông hơi', 'Derived amenities'),
    ('partner', 'accommodation', 'general_amenities', 'Bồn tắm nóng (Jacuzzi)', 'Jacuzzi', 'Bồn tắm nóng', None),
    ('partner', 'accommodation', 'general_amenities', 'Sân vườn', 'Garden', 'Sân vườn', None),
    ('google', 'accommodation', 'general_amenities', 'garden', 'Garden', 'Sân vườn', 'Derived amenities'),
    ('google', 'accommodation', 'general_amenities', 'botanical_garden', 'Garden', 'Sân vườn', 'Derived amenities'),
    ('partner', 'accommodation', 'general_amenities', 'Phòng họp', 'Meeting Rooms', 'Phòng họp', None),
    ('google', 'accommodation', 'general_amenities', 'meeting_rooms', 'Meeting Rooms', 'Phòng họp', None),
    ('partner', 'accommodation', 'general_amenities', 'Khu vực làm việc chung', 'Co-working Space', 'Khu làm việc chung', None),
    ('partner', 'accommodation', 'general_amenities', 'Giữ hành lý', 'Luggage Storage', 'Giữ hành lý', None),
    ('partner', 'accommodation', 'general_amenities', 'Thu đổi ngoại tệ', 'Currency Exchange', 'Thu đổi ngoại tệ', None),
    ('partner', 'accommodation', 'general_amenities', 'Trạm sạc xe điện', 'EV Charging Station', 'Trạm sạc xe điện', None),
    ('google', 'accommodation', 'general_amenities', 'parkingOptions.freeElectricVehicleParking', 'EV Charging Station', 'Trạm sạc xe điện', 'Parking'),
    ('google', 'accommodation', 'general_amenities', 'parkingOptions.paidElectricVehicleParking', 'EV Charging Station', 'Trạm sạc xe điện', 'Parking'),
    ('google', 'accommodation', 'general_amenities', "json_type = 'electric_vehicle_charging_station'", 'EV Charging Station', 'Trạm sạc xe điện', 'Derived amenities'),
    ('google', 'accommodation', 'accessibility', 'accessibilityOptions.wheelchairAccessibleSeating', 'Wheelchair Accessible Seating', 'Chỗ ngồi cho người khuyết tật', 'Accessibility'),
    ('google', 'accommodation', 'accessibility', 'accessibilityOptions.wheelchairAccessibleElevator', 'Wheelchair Accessible Elevator', 'Thang máy hỗ trợ xe lăn', 'Accessibility'),
    ('partner', 'accommodation', 'accessibility', 'Lối đi xe lăn', 'Wheelchair Accessible Entrance', 'Lối đi xe lăn', None),
    ('google', 'accommodation', 'accessibility', 'accessibilityOptions.wheelchairAccessibleEntrance', 'Wheelchair Accessible Entrance', 'Lối đi xe lăn', 'Accessibility'),
    ('partner', 'accommodation', 'accessibility', 'Thang máy có chữ nổi', 'Braille Elevator', 'Thang máy có chữ nổi', None),
    ('partner', 'accommodation', 'accessibility', 'Nhà vệ sinh có tay vịn', 'Wheelchair Accessible Restroom', 'Nhà vệ sinh có tay vịn', None),
    ('google', 'accommodation', 'accessibility', 'accessibilityOptions.wheelchairAccessibleRestroom', 'Wheelchair Accessible Restroom', 'Nhà vệ sinh có tay vịn', 'Accessibility'),
    ('partner', 'accommodation', 'accessibility', 'Bồn rửa mặt thấp', 'Lowered Sink', 'Bồn rửa mặt thấp', None),
    ('partner', 'accommodation', 'accessibility', 'Bãi đỗ xe cho người khuyết tật', 'Accessible Parking', 'Bãi đỗ xe cho người khuyết tật', None),
    ('google', 'accommodation', 'accessibility', 'accessibilityOptions.wheelchairAccessibleParking', 'Accessible Parking', 'Bãi đỗ xe cho người khuyết tật', 'Accessibility'),
    ('google', 'f_n_b', 'specialty_tags', 'dim_sum_restaurant', 'Dim Sum', 'Dimsum', None),
    ('google', 'f_n_b', 'specialty_tags', 'sushi_restaurant', 'Sushi', 'Sushi', None),
    ('google', 'f_n_b', 'specialty_tags', 'dumpling_restaurant', 'Dumpling', 'Há cảo – Bánh bao', None),
    ('google', 'f_n_b', 'specialty_tags', 'tonkatsu_restaurant', 'Tonkatsu', 'Tonkatsu', None),
    ('google', 'f_n_b', 'specialty_tags', 'japanese_curry_restaurant', 'Japanese Curry', 'Cà ri Nhật', None),
    ('google', 'f_n_b', 'specialty_tags', 'cantonese_restaurant', 'Cantonese', 'Ẩm thực Quảng Đông', None),
    ('google', 'f_n_b', 'specialty_tags', 'chinese_noodle_restaurant', 'Chinese Noodle', 'Mì Trung Quốc', None),
    ('google', 'f_n_b', 'specialty_tags', 'ramen_restaurant', 'Ramen', 'Ramen', None),
    ('google', 'f_n_b', 'specialty_tags', 'yakiniku_restaurant, yakitori_restaurant', 'Yakiniku / Teppanyaki', 'Yakiniku / Teppanyaki', None),
    ('google', 'f_n_b', 'specialty_tags', 'japanese_izakaya_restaurant', 'Izakaya', 'Izakaya', None),
    ('google', 'f_n_b', 'specialty_tags', 'korean_barbecue_restaurant', 'Korean BBQ', 'BBQ Hàn Quốc', None),
    ('google', 'f_n_b', 'specialty_tags', 'pizza_restaurant', 'Pizza', 'Pizza', None),
    ('google', 'f_n_b', 'specialty_tags', 'hamburger_restaurant', 'Hamburger', 'Hamburger', None),
    ('google', 'f_n_b', 'specialty_tags', 'brunch_restaurant', 'Brunch', 'Brunch', None),
    ('google', 'f_n_b', 'specialty_tags', 'cat_cafe, dog_cafe', 'Pet Café', 'Quán cà phê thú cưng', None),
    ('google', 'f_n_b', 'specialty_tags', 'tapas_restaurant', 'Tapas', 'Tapas', None),
    ('google', 'f_n_b', 'specialty_tags', 'soup_restaurant', 'Soup', 'Súp', None),
    ('partner', 'f_n_b', 'specialty_tags', 'Street Food', 'Street Food', 'Ẩm thực đường phố', None),
    ('google', 'f_n_b', 'specialty_tags', 'food_stand', 'Street Food', 'Ẩm thực đường phố', None),
    ('google', 'f_n_b', 'specialty_tags', 'street_food_gathering', 'Street Food', 'Ẩm thực đường phố', None),
    ('partner', 'f_n_b', 'specialty_tags', 'Seafood', 'Seafood', 'Hải sản', None),
    ('google', 'f_n_b', 'specialty_tags', 'seafood_restaurant', 'Seafood', 'Hải sản', None),
    ('google', 'f_n_b', 'specialty_tags', 'oyster_bar_restaurant', 'Seafood', 'Hải sản', None),
    ('partner', 'f_n_b', 'specialty_tags', 'Steakhouse', 'Steakhouse', 'Bít tết', None),
    ('google', 'f_n_b', 'specialty_tags', 'steakhouse', 'Steakhouse', 'Bít tết', None),
    ('partner', 'f_n_b', 'specialty_tags', 'Office Lunch', 'Office Lunch', 'Cơm văn phòng', None),
    ('partner', 'f_n_b', 'specialty_tags', 'Milk Tea', 'Milk Tea', 'Trà sữa', None),
    ('partner', 'f_n_b', 'specialty_tags', 'Bakery', 'Bakery', 'Tiệm bánh', None),
    ('google', 'f_n_b', 'specialty_tags', 'bakery', 'Bakery', 'Tiệm bánh', None),
    ('google', 'f_n_b', 'specialty_tags', 'pastry_shop', 'Bakery', 'Tiệm bánh', None),
    ('google', 'f_n_b', 'specialty_tags', 'cake_shop', 'Bakery', 'Tiệm bánh', None),
    ('partner', 'f_n_b', 'specialty_tags', 'Bar-Pub', 'Bar & Pub', 'Bar – Pub', None),
    ('google', 'f_n_b', 'specialty_tags', "json_type in ('bar', 'pub', 'night_club', 'cocktail_bar', 'wine_bar', 'lounge_bar', 'sports_bar', 'hookah_bar', 'irish_pub', 'beer_garden', 'karaoke')", 'Bar & Pub', 'Bar – Pub', 'Derived amenities'),
    ('partner', 'f_n_b', 'specialty_tags', 'Rooftop Bar', 'Rooftop Bar', 'Bar sân thượng', None),
    ('google', 'f_n_b', 'space_and_services', 'restroom', 'Restroom', 'Nhà vệ sinh', 'Suitability'),
    ('google', 'f_n_b', 'space_and_services', 'goodForChildren', 'Good for Children', 'Thân thiện trẻ em', 'Suitability'),
    ('partner', 'f_n_b', 'space_and_services', 'Outdoor Seating', 'Outdoor Seating', 'Chỗ ngồi ngoài trời', None),
    ('google', 'f_n_b', 'space_and_services', 'outdoorSeating = true = true', 'Outdoor Seating', 'Chỗ ngồi ngoài trời', None),
    ('partner', 'f_n_b', 'space_and_services', 'Private Room (VIP)', 'Private Room (VIP)', 'Phòng riêng (VIP)', None),
    ('partner', 'f_n_b', 'space_and_services', 'Scenic View', 'Scenic View', 'Góc nhìn đẹp', None),
    ('partner', 'f_n_b', 'space_and_services', 'Romantic Decor', 'Romantic Decor', 'Trang trí lãng mạn', None),
    ('partner', 'f_n_b', 'space_and_services', 'High Chair', 'High Chair', 'Có ghế trẻ em', None),
    ('partner', 'f_n_b', 'space_and_services', 'Wheelchair Accessible', 'Wheelchair Accessible', 'Lối đi xe lăn', None),
    ('google', 'f_n_b', 'space_and_services', 'accessibilityOptions.wheelchairAccessibleEntrance', 'Wheelchair Accessible', 'Lối đi xe lăn', 'Accessibility'),
    ('partner', 'f_n_b', 'space_and_services', 'Sports Screening', 'Sports Screening', 'Chiếu thể thao trực tiếp', None),
    ('google', 'f_n_b', 'space_and_services', 'goodForWatchingSports', 'Sports Screening', 'Chiếu thể thao trực tiếp', 'Suitability'),
    ('partner', 'f_n_b', 'space_and_services', 'Live Music', 'Live Music', 'Nhạc sống', None),
    ('google', 'f_n_b', 'space_and_services', 'liveMusic', 'Live Music', 'Nhạc sống', 'Food/service'),
    ('partner', 'f_n_b', 'space_and_services', 'Car Parking', 'Car Parking', 'Bãi đỗ xe ô tô', None),
    ('google', 'f_n_b', 'space_and_services', 'parkingOptions.freeParkingLot, parkingOptions.paidParkingLot = true', 'Car Parking', 'Bãi đỗ xe ô tô', 'Parking'),
    ('partner', 'f_n_b', 'space_and_services', 'Table Service', 'Table Service', 'Phục vụ tại bàn', None),
    ('google', 'f_n_b', 'space_and_services', 'dineIn', 'Table Service', 'Phục vụ tại bàn', 'Food/service'),
    ('google', 'attraction', 'facilities_and_services', 'accessibilityOptions.wheelchairAccessibleEntrance', 'Wheelchair Accessible Entrance', 'Lối đi xe lăn', 'Accessibility'),
    ('google', 'attraction', 'facilities_and_services', 'accessibilityOptions.wheelchairAccessibleParking', 'Accessible Parking', 'Bãi đỗ xe người khuyết tật', 'Accessibility'),
    ('google', 'attraction', 'facilities_and_services', 'accessibilityOptions.wheelchairAccessibleRestroom', 'Wheelchair Accessible Restroom', 'Nhà vệ sinh xe lăn', 'Accessibility'),
    ('google', 'attraction', 'facilities_and_services', 'restroom', 'Restroom', 'Nhà vệ sinh', 'Suitability'),
    ('google', 'attraction', 'facilities_and_services', 'goodForChildren', 'Good for Children', 'Thân thiện trẻ em', 'Suitability'),
    ('google', 'attraction', 'facilities_and_services', 'liveMusic', 'Live Music', 'Nhạc Live', 'Food/service'),
    ('partner', 'attraction', 'facilities_and_services', 'Indoor', 'Indoor', 'Trong nhà', None),
    ('partner', 'attraction', 'facilities_and_services', 'Outdoor', 'Outdoor', 'Ngoài trời', None),
    ('partner', 'attraction', 'facilities_and_services', 'Covered', 'Covered', 'Có mái che', None),
    ('partner', 'attraction', 'facilities_and_services', 'Family Friendly', 'Family Friendly', 'Thân thiện với gia đình', None),
    ('google', 'attraction', 'facilities_and_services', 'goodForChildren', 'Family Friendly', 'Thân thiện với gia đình', 'Suitability'),
    ('partner', 'attraction', 'facilities_and_services', 'Couples', 'Couples', 'Cặp đôi', None),
    ('partner', 'attraction', 'facilities_and_services', 'Group', 'Group', 'Nhóm bạn', None),
    ('google', 'attraction', 'facilities_and_services', 'goodForGroups', 'Group', 'Nhóm bạn', 'Suitability'),
    ('partner', 'attraction', 'facilities_and_services', 'History', 'History', 'Lịch sử', None),
    ('google', 'attraction', 'facilities_and_services', "json_type in ('historical_landmark', 'archaeological_site', 'ruins', 'castle', 'palace', 'monument', 'sculpture', 'lighthouse')", 'History', 'Lịch sử', None),
    ('partner', 'attraction', 'facilities_and_services', 'Art', 'Art', 'Nghệ thuật', None),
    ('google', 'attraction', 'facilities_and_services', "json_type = 'art_gallery'", 'Art', 'Nghệ thuật', None),
    ('partner', 'attraction', 'facilities_and_services', 'Science', 'Science', 'Khoa học', None),
    ('google', 'attraction', 'facilities_and_services', "json_type = 'planetarium'", 'Science', 'Khoa học', None),
    ('partner', 'attraction', 'facilities_and_services', 'Interactive', 'Interactive', 'Tương tác', None),
    ('partner', 'attraction', 'facilities_and_services', 'Educational', 'Educational', 'Giáo dục', None),
    ('partner', 'attraction', 'facilities_and_services', 'Spiritual', 'Spiritual', 'Tâm linh', None),
    ('google', 'attraction', 'facilities_and_services', "json_type in ('church', 'hindu_temple', 'mosque', 'synagogue', 'buddhist_temple', 'place_of_worship')", 'Spiritual', 'Tâm linh', None),
    ('partner', 'attraction', 'facilities_and_services', 'Parking', 'Parking', 'Bãi đậu xe', None),
    ('google', 'attraction', 'facilities_and_services', 'parkingOptions.freeParkingLot', 'Parking', 'Bãi đậu xe', 'Parking'),
    ('google', 'attraction', 'facilities_and_services', 'parkingOptions.paidParkingLot', 'Parking', 'Bãi đậu xe', 'Parking'),
    ('partner', 'attraction', 'facilities_and_services', 'Free Wi-Fi', 'Free Wi-Fi', 'Wi-Fi miễn phí', None),
    ('partner', 'attraction', 'facilities_and_services', 'Lockers', 'Lockers', 'Tủ gửi đồ', None),
    ('partner', 'attraction', 'facilities_and_services', 'Restaurant-Cafe', 'Restaurant-Cafe', 'Nhà hàng – Quán ăn', None),
    ('google', 'attraction', 'facilities_and_services', "json_type in ('restaurant', 'cafe', 'coffee_shop')", 'Restaurant-Cafe', 'Nhà hàng – Quán ăn', 'Derived amenities'),
    ('partner', 'attraction', 'facilities_and_services', 'Gift Shop', 'Gift Shop', 'Cửa hàng lưu niệm', None),
    ('google', 'attraction', 'facilities_and_services', "json_type = 'gift_shop'", 'Gift Shop', 'Cửa hàng lưu niệm', None),
    ('partner', 'attraction', 'facilities_and_services', 'Stroller Rental', 'Stroller Rental', 'Cho thuê xe đẩy trẻ em', None),
    ('partner', 'attraction', 'facilities_and_services', 'Wheelchair Rental', 'Wheelchair Rental', 'Cho thuê xe lăn', None),
    ('partner', 'attraction', 'facilities_and_services', 'First Aid Room', 'First Aid Room', 'Phòng y tế', None),
    ('partner', 'attraction', 'facilities_and_services', 'On-site Tour Guide', 'On-site Tour Guide', 'Hướng dẫn viên tại điểm', None),
    ('partner', 'attraction', 'facilities_and_services', 'Audio Guide', 'Audio Guide', 'Máy hướng dẫn tự động', None),
    ('partner', 'shopping_n_retail', 'facilities_and_services', 'Tax Refund', 'Tax Refund', 'Hoàn thuế', None),
    ('partner', 'shopping_n_retail', 'facilities_and_services', 'Shipping-Delivery', 'Shipping-Delivery', 'Giao hàng tận nơi', None),
    ('google', 'shopping_n_retail', 'facilities_and_services', 'delivery', 'Shipping-Delivery', 'Giao hàng tận nơi', 'Food/service'),
    ('partner', 'shopping_n_retail', 'facilities_and_services', 'Gift Wrapping', 'Gift Wrapping', 'Gói quà miễn phí', None),
    ('partner', 'shopping_n_retail', 'facilities_and_services', 'Fitting Room', 'Fitting Room', 'Phòng thử đồ', None),
    ('partner', 'shopping_n_retail', 'facilities_and_services', 'Personalization-Engraving', 'Personalization-Engraving', 'Khắc tên – Cá nhân hóa', None),
    ('partner', 'shopping_n_retail', 'facilities_and_services', 'Tailoring', 'Tailoring', 'May đo theo yêu cầu', None),
    ('partner', 'shopping_n_retail', 'facilities_and_services', 'Credit Card (Visa/Master)', 'Credit Card', 'Chấp nhận thẻ tín dụng', None),
    ('google', 'shopping_n_retail', 'facilities_and_services', 'paymentOptions.acceptsCreditCards', 'Credit Card', 'Chấp nhận thẻ tín dụng', 'Payment'),
    ('partner', 'shopping_n_retail', 'facilities_and_services', 'E-Wallet', 'E-Wallet', 'Ví điện tử', None),
    ('partner', 'shopping_n_retail', 'facilities_and_services', 'VAT Invoice', 'VAT Invoice', 'Xuất hóa đơn VAT', None),
    ('partner', 'shopping_n_retail', 'facilities_and_services', 'Flexible Return', 'Flexible Return', 'Đổi trả linh hoạt', None),
    ('partner', 'shopping_n_retail', 'facilities_and_services', 'Parking', 'Parking', 'Bãi đỗ xe', None),
    ('google', 'shopping_n_retail', 'facilities_and_services', 'parkingOptions.freeParkingLot, parkingOptions.paidParkingLot', 'Parking', 'Bãi đỗ xe', 'Parking'),
    ('partner', 'shopping_n_retail', 'facilities_and_services', 'Free Wi-Fi', 'Free Wi-Fi', 'Wi-Fi miễn phí', None),
    ('google', 'other', 'other', 'takeout', 'takeout', 'takeout', 'Food/service'),
    ('google', 'other', 'other', 'curbsidePickup', 'curbsidePickup', 'curbsidePickup', 'Food/service'),
    ('google', 'other', 'other', 'reservable', 'reservable', 'reservable', 'Food/service'),
    ('google', 'other', 'other', 'servesBreakfast', 'servesBreakfast', 'servesBreakfast', 'Food/service'),
    ('google', 'other', 'other', 'servesLunch', 'servesLunch', 'servesLunch', 'Food/service'),
    ('google', 'other', 'other', 'servesDinner', 'servesDinner', 'servesDinner', 'Food/service'),
    ('google', 'other', 'other', 'servesBeer', 'servesBeer', 'servesBeer', 'Food/service'),
    ('google', 'other', 'other', 'servesWine', 'servesWine', 'servesWine', 'Food/service'),
    ('google', 'other', 'other', 'servesBrunch', 'servesBrunch', 'servesBrunch', 'Food/service'),
    ('google', 'other', 'other', 'servesVegetarianFood', 'servesVegetarianFood', 'servesVegetarianFood', 'Food/service'),
    ('google', 'other', 'other', 'menuForChildren', 'menuForChildren', 'menuForChildren', 'Food/service'),
    ('google', 'other', 'other', 'servesDessert', 'servesDessert', 'servesDessert', 'Food/service'),
    ('google', 'other', 'other', 'servesCoffee', 'servesCoffee', 'servesCoffee', 'Food/service'),
    ('google', 'other', 'other', 'allowsDogs', 'allowsDogs', 'allowsDogs', 'Suitability'),
    ('google', 'other', 'other', 'pureServiceAreaBusiness', 'pureServiceAreaBusiness', 'pureServiceAreaBusiness', 'Suitability'),
    ('google', 'other', 'other', 'acceptsDebitCards', 'acceptsDebitCards', 'acceptsDebitCards', 'Payment'),
    ('google', 'other', 'other', 'acceptsNfc', 'acceptsNfc', 'acceptsNfc', 'Payment'),
]

def validate_seed(rows):
    """Seed phải đúng nguồn, sector, nhóm, khoá 5 cột, chỉ còn dạng key; khoá 4 cột trùng chỉ trong AMENITY_SHARED_RAW."""
    keys = [row[:5] for row in rows]
    if len(keys) != len(set(keys)):
        raise ValueError("ref_dictionary_amenities: trùng khoá 5 cột")
    counts = {}
    for row in rows:
        counts[row[:4]] = counts.get(row[:4], 0) + 1
    dup = sorted(key for key, n in counts.items() if n > 1 and key not in AMENITY_SHARED_RAW)
    if dup:
        raise ValueError(f"ref_dictionary_amenities: trùng amenity_raw trong (source, sector, group): {dup}")
    allowed_flags = {
        "Food/service", "Suitability", "Payment", "Parking",
        "Accessibility", "Derived amenities",
    }
    for source, sector, group, raw, amenity, _vi, flags in rows:
        if source not in {"google", "partner"}:
            raise ValueError(f"source lạ: {source}")
        if sector not in SEED_SECTORS:
            raise ValueError(f"sector lạ: {sector}")
        if group not in SEED_GROUPS:
            raise ValueError(f"group lạ: {group}")
        if not raw or raw in SEED_SKIP:
            raise ValueError(f"amenity_raw không hợp lệ: {raw}")
        if amenity_keys(raw) != [raw]:
            raise ValueError(f"amenity_raw còn dạng cũ: {raw}")
        if not amenity:
            raise ValueError(f"amenity trống: {raw}")
        if flags not in allowed_flags and flags is not None:
            raise ValueError(f"service_flags lạ: {flags!r}")
# END_AMENITY_SEED
validate_seed(SEED_ROWS)
AMENITY_ROWS = list(SEED_ROWS)


In [ ]:
import json

apply = to_bool(p_apply)
errors = []
partner_rows = flatten_table_configs(TABLE_CONFIGS, "lh_vv_bronze.dbo", "partner_raw_data")
third_rows = flatten_table_configs(TABLE_CONFIGS_3P, "lh_vv_bronze.dbo", "brz_3rd_crawler_poi_stream")
try:
    validate_rows(TABLE_CONFIGS, partner_rows)
    validate_rows(TABLE_CONFIGS_3P, third_rows)
except ValueError as exc:
    errors.append(str(exc))
registry_rows = partner_rows + third_rows
seen = set()
for row in registry_rows:
    key = (row["src_schema"], row["src_tbl"], row["trg_schema"], row["trg_tbl"], row["trg_column"])
    if key in seen:
        errors.append(f"registry trùng {key}")
    seen.add(key)
pc_keys = set()
for pl, src_schema, src_tbl, trg_schema, trg_tbl, *_rest in PIPELINE_CONFIG_ROWS:
    key = (pl, src_schema, src_tbl, trg_schema, trg_tbl)
    if key in pc_keys:
        errors.append(f"pipeline_config trùng {key}")
    pc_keys.add(key)
wm_ids = [row[0] for row in WATERMARK_ROWS]
if len(wm_ids) != len(set(wm_ids)):
    errors.append(f"watermark_id trùng {wm_ids}")
fqs = [spec["fq"] for spec in TABLE_SPECS]
if len(fqs) != len(set(fqs)) or len(fqs) != 16:
    errors.append(f"TABLE_SPECS cần 16 fq duy nhất, có {len(fqs)}")
if errors:
    raise ConfigError("\n".join(errors))

for db in ("lh_vv_ctrl.dbo", "lh_vv_silver.dbo", "lh_vv_gold.dbo"):
    try:
        spark.sql(f"SHOW TABLES IN {db}")
    except Exception as exc:
        raise ConfigError(f"thiếu lakehouse/schema {db}: {exc}") from exc

schema_issues = []
table_actions = []
for spec in TABLE_SPECS:
    action = "EXISTS_OK" if table_exists(spec["fq"]) else "CREATE"
    table_actions.append({"fq": spec["fq"], "action": action})
    print(f"{spec['fq']} {action}")
    print(create_sql(spec))
    if action == "EXISTS_OK":
        fields = spark.table(spec["fq"]).schema.fields
        parts = list(spark.sql(f"DESCRIBE DETAIL {spec['fq']}").first()["partitionColumns"] or [])
        schema_issues += [f"{spec['fq']}: {item}" for item in schema_diff(spec, fields, parts)]

pc_df = spark.createDataFrame(
    [(i, *row) for i, row in enumerate(PIPELINE_CONFIG_ROWS, start=1)],
    "seq INT, pl_name STRING, src_schema STRING, src_tbl STRING, trg_schema STRING, trg_tbl STRING, "
    "is_active INT, priority INT, load_mode STRING, align_path STRING, dedup_order STRING, nb_name STRING",
)
pc_df.createOrReplaceTempView("_v_setup_pipeline_config")
wm_df = spark.createDataFrame(
    WATERMARK_ROWS,
    "watermark_id STRING, src_schema STRING, src_tbl STRING, watermark_column STRING, "
    "status STRING, flow_name STRING, watermark_sequence LONG",
)
wm_df.createOrReplaceTempView("_v_setup_watermark")
spark.createDataFrame(registry_rows, schema=REGISTRY_SCHEMA).createOrReplaceTempView("_v_setup_registry")
spark.createDataFrame(
    PLACE_TYPE_ROWS,
    "source_name STRING, place_type STRING, business_sector STRING, business_category STRING, business_category_vi STRING, category_group STRING, range_low_min INT, range_high_min INT, avg_range_min INT, candidate_best_time_code STRING, canonical_candidate STRING",
).createOrReplaceTempView("_v_setup_place_type")
spark.createDataFrame(
    AMENITY_ROWS,
    "source_name STRING, business_sector STRING, amenity_group STRING, amenity_raw STRING, amenity STRING, amenity_vi STRING, service_flags STRING",
).createOrReplaceTempView("_v_setup_amenities")

SEED_PLAN = [
    ("pipeline_config", "_v_setup_pipeline_config", "lh_vv_ctrl.dbo.ctrl_mng_pipeline_config",
     "t.pl_name = s.pl_name AND t.src_schema = s.src_schema AND t.src_tbl = s.src_tbl "
     "AND t.trg_schema = s.trg_schema AND t.trg_tbl = s.trg_tbl", len(PIPELINE_CONFIG_ROWS)),
    ("watermark", "_v_setup_watermark", "lh_vv_ctrl.dbo.ctrl_mng_watermark",
     "t.watermark_id = s.watermark_id", len(WATERMARK_ROWS)),
    ("registry", "_v_setup_registry", "lh_vv_ctrl.dbo.ctrl_cfg_schema_registry",
     "t.src_schema = s.src_schema AND t.src_tbl = s.src_tbl AND t.trg_schema = s.trg_schema "
     "AND t.trg_tbl = s.trg_tbl AND t.trg_column = s.trg_column", len(registry_rows)),
    ("place_type", "_v_setup_place_type", "lh_vv_silver.dbo.ref_dictionary_place_type",
     "t.source_name = s.source_name AND t.place_type = s.place_type", len(PLACE_TYPE_ROWS)),
    ("amenities", "_v_setup_amenities", "lh_vv_silver.dbo.ref_dictionary_amenities",
     "t.source_name = s.source_name AND t.business_sector = s.business_sector "
     "AND t.amenity_group = s.amenity_group AND t.amenity_raw = s.amenity_raw AND t.amenity = s.amenity",
     len(AMENITY_ROWS)),
]


def _missing(view, table, on_sql, defined):
    if not table_exists(table):
        return defined
    return int(spark.sql(
        f"SELECT count(*) AS n FROM {view} s LEFT ANTI JOIN {table} t ON {on_sql}"
    ).first()["n"])


def _dup_keys(view, keys):
    group = ", ".join(keys)
    return int(spark.sql(
        f"SELECT count(*) AS n FROM (SELECT {group} FROM {view} GROUP BY {group} HAVING count(*) > 1)"
    ).first()["n"])


def _target_dup_keys(table, keys):
    """Khoá logic đã trùng trên bảng đích thì dừng trước khi ghi."""
    if not table_exists(table):
        return 0
    group = ", ".join(keys)
    return int(spark.sql(
        f"SELECT count(*) AS n FROM (SELECT {group} FROM {table} GROUP BY {group} HAVING count(*) > 1)"
    ).first()["n"])


SEED_KEYS = {
    "pipeline_config": ["pl_name", "src_schema", "src_tbl", "trg_schema", "trg_tbl"],
    "watermark": ["watermark_id"],
    "registry": ["src_schema", "src_tbl", "trg_schema", "trg_tbl", "trg_column"],
    "place_type": ["source_name", "place_type"],
    "amenities": ["source_name", "business_sector", "amenity_group", "amenity_raw", "amenity"],
}
seed_counts = {}
for name, view, table, on_sql, defined in SEED_PLAN:
    dups = _dup_keys(view, SEED_KEYS[name])
    target_dups = _target_dup_keys(table, SEED_KEYS[name])
    missing = _missing(view, table, on_sql, defined)
    seed_counts[name] = {"defined": defined, "to_insert": missing, "dup_keys": dups, "target_dups": target_dups}
    if dups:
        schema_issues.append(f"{name}: {dups} khoá seed trùng")
    if target_dups:
        schema_issues.append(f"{name}: {target_dups} khoá trùng trên {table}")
if schema_issues:
    raise ConfigError("\n".join(schema_issues))
for name, counts in seed_counts.items():
    print(f"seed={name} defined={counts['defined']} to_insert={counts['to_insert']}")
if not apply:
    notebookutils.notebook.exit(json.dumps({
        "status": "PLAN",
        "p_apply": False,
        "tables": table_actions,
        "seeds": {name: {"defined": item["defined"], "to_insert": item["to_insert"]} for name, item in seed_counts.items()},
    }))


In [ ]:
def _create_enr_schema():
    spark.sql("CREATE SCHEMA IF NOT EXISTS lh_vv_gold.enr")


with_retry(_create_enr_schema, "schema lh_vv_gold.enr")
for spec in TABLE_SPECS:
    def _create(spec=spec):
        spark.sql(create_sql(spec))
    with_retry(_create, spec["fq"])

post_issues = []
for spec in TABLE_SPECS:
    if not table_exists(spec["fq"]):
        post_issues.append(f"thiếu bảng {spec['fq']}")
        continue
    fields = spark.table(spec["fq"]).schema.fields
    parts = list(spark.sql(f"DESCRIBE DETAIL {spec['fq']}").first()["partitionColumns"] or [])
    post_issues += [f"{spec['fq']}: {item}" for item in schema_diff(spec, fields, parts)]
if post_issues:
    raise ConfigError("\n".join(post_issues))


In [ ]:
PC_FQ = "lh_vv_ctrl.dbo.ctrl_mng_pipeline_config"
WM_FQ = "lh_vv_ctrl.dbo.ctrl_mng_watermark"
REG_FQ = "lh_vv_ctrl.dbo.ctrl_cfg_schema_registry"
PLACE_FQ = "lh_vv_silver.dbo.ref_dictionary_place_type"
AMENITY_FQ = "lh_vv_silver.dbo.ref_dictionary_amenities"
SEED_RESULTS = {}


def _merge_pipeline_config():
    base = int(spark.sql(f"SELECT COALESCE(MAX(id), 0) AS m FROM {PC_FQ}").first()["m"])
    spark.sql(f"""
        MERGE INTO {PC_FQ} AS t
        USING _v_setup_pipeline_config AS s
        ON t.pl_name = s.pl_name AND t.src_schema = s.src_schema AND t.src_tbl = s.src_tbl
           AND t.trg_schema = s.trg_schema AND t.trg_tbl = s.trg_tbl
        WHEN NOT MATCHED THEN INSERT
            (id, pl_name, src_schema, src_tbl, trg_schema, trg_tbl, is_active, priority,
             load_mode, align_path, dedup_order, nb_name)
        VALUES ({base} + s.seq, s.pl_name, s.src_schema, s.src_tbl, s.trg_schema, s.trg_tbl, s.is_active,
                s.priority, s.load_mode, s.align_path, s.dedup_order, s.nb_name)
    """)


def _merge_watermark():
    spark.sql(f"""
        MERGE INTO {WM_FQ} AS t
        USING _v_setup_watermark AS s
        ON t.watermark_id = s.watermark_id
        WHEN NOT MATCHED THEN INSERT (
            watermark_id, src_schema, src_tbl, trg_schema, trg_tbl,
            watermark_column, watermark_value, last_success_at, status, flow_name,
            watermark_sequence, last_run_id, last_src_version, last_src_table_id, error_message,
            lock_exec_id, lock_at, updated_at
        ) VALUES (
            s.watermark_id, s.src_schema, s.src_tbl, CAST(NULL AS STRING), CAST(NULL AS STRING),
            s.watermark_column, CAST(NULL AS TIMESTAMP), CAST(NULL AS TIMESTAMP), s.status, s.flow_name,
            s.watermark_sequence, CAST(NULL AS STRING), CAST(NULL AS BIGINT), CAST(NULL AS STRING),
            CAST(NULL AS STRING), CAST(NULL AS STRING), CAST(NULL AS TIMESTAMP), current_timestamp()
        )
    """)


def _merge_registry():
    spark.sql(f"""
        MERGE INTO {REG_FQ} AS t
        USING _v_setup_registry AS s
        ON t.src_schema = s.src_schema AND t.src_tbl = s.src_tbl
           AND t.trg_schema = s.trg_schema AND t.trg_tbl = s.trg_tbl AND t.trg_column = s.trg_column
        WHEN NOT MATCHED THEN INSERT (
            src_schema, src_tbl, src_object_schema, src_object, trg_schema, trg_tbl, trg_column,
            json_path, data_type, convert_arg, convert_rule, is_key, key_order, is_toast,
            column_order, is_active, description
        ) VALUES (
            s.src_schema, s.src_tbl, s.src_object_schema, s.src_object, s.trg_schema, s.trg_tbl, s.trg_column,
            s.json_path, s.data_type, s.convert_arg, s.convert_rule, s.is_key, s.key_order, s.is_toast,
            s.column_order, s.is_active, s.description
        )
    """)


def _merge_place_type():
    spark.sql(f"""
        MERGE INTO {PLACE_FQ} AS t
        USING _v_setup_place_type AS s
        ON t.source_name = s.source_name AND t.place_type = s.place_type
        WHEN NOT MATCHED THEN INSERT
            (source_name, place_type, business_sector, business_category, business_category_vi, is_active, updated_at, category_group, range_low_min, range_high_min, avg_range_min, candidate_best_time_code, canonical_candidate)
        VALUES (s.source_name, s.place_type, s.business_sector, s.business_category, s.business_category_vi,
                true, current_timestamp(), s.category_group, s.range_low_min, s.range_high_min, s.avg_range_min, s.candidate_best_time_code, s.canonical_candidate)
    """)


def _merge_amenities():
    spark.sql(f"""
        MERGE INTO {AMENITY_FQ} AS t
        USING _v_setup_amenities AS s
        ON t.source_name = s.source_name AND t.business_sector = s.business_sector
           AND t.amenity_group = s.amenity_group AND t.amenity_raw = s.amenity_raw AND t.amenity = s.amenity
        WHEN NOT MATCHED THEN INSERT (
            source_name, business_sector, amenity_group, amenity_raw, amenity, amenity_vi, service_flags, is_active, updated_at
        ) VALUES (
            s.source_name, s.business_sector, s.amenity_group, s.amenity_raw, s.amenity, s.amenity_vi, s.service_flags,
            true, current_timestamp()
        )
    """)


def _apply_seed(name, view, table, on_sql, merge_fn, drift_sql):
    defined = int(spark.sql(f"SELECT count(*) AS n FROM {view}").first()["n"])
    before = _missing(view, table, on_sql, defined)
    with_retry(merge_fn, f"seed {name}")
    after = _missing(view, table, on_sql, defined)
    if after:
        raise ConfigError(f"{name}: còn thiếu {after} khoá sau MERGE")
    drift = int(spark.sql(drift_sql).first()["n"]) if table_exists(table) else 0
    inserted = before - after
    SEED_RESULTS[name] = {"defined": defined, "inserted": inserted, "drift": drift}
    print(f"seed={name} inserted={inserted} drift={drift}")


_apply_seed(
    "pipeline_config", "_v_setup_pipeline_config", PC_FQ,
    "t.pl_name = s.pl_name AND t.src_schema = s.src_schema AND t.src_tbl = s.src_tbl "
    "AND t.trg_schema = s.trg_schema AND t.trg_tbl = s.trg_tbl",
    _merge_pipeline_config,
    f"""
    SELECT count(*) AS n
    FROM _v_setup_pipeline_config s
    JOIN {PC_FQ} t
      ON t.pl_name = s.pl_name AND t.src_schema = s.src_schema AND t.src_tbl = s.src_tbl
     AND t.trg_schema = s.trg_schema AND t.trg_tbl = s.trg_tbl
    WHERE NOT (
        t.is_active <=> s.is_active AND t.priority <=> s.priority AND t.load_mode <=> s.load_mode
        AND t.align_path <=> s.align_path AND t.dedup_order <=> s.dedup_order AND t.nb_name <=> s.nb_name
    )
    """,
)
_apply_seed(
    "watermark", "_v_setup_watermark", WM_FQ, "t.watermark_id = s.watermark_id", _merge_watermark,
    f"""
    SELECT count(*) AS n
    FROM _v_setup_watermark s
    JOIN {WM_FQ} t ON t.watermark_id = s.watermark_id
    WHERE NOT (
        t.src_schema <=> s.src_schema AND t.src_tbl <=> s.src_tbl
        AND t.watermark_column <=> s.watermark_column AND t.status <=> s.status
        AND t.flow_name <=> s.flow_name AND t.watermark_sequence <=> s.watermark_sequence
    )
    """,
)
_apply_seed(
    "registry", "_v_setup_registry", REG_FQ,
    "t.src_schema = s.src_schema AND t.src_tbl = s.src_tbl AND t.trg_schema = s.trg_schema "
    "AND t.trg_tbl = s.trg_tbl AND t.trg_column = s.trg_column",
    _merge_registry,
    f"""
    SELECT count(*) AS n
    FROM _v_setup_registry s
    JOIN {REG_FQ} t
      ON t.src_schema = s.src_schema AND t.src_tbl = s.src_tbl
     AND t.trg_schema = s.trg_schema AND t.trg_tbl = s.trg_tbl AND t.trg_column = s.trg_column
    WHERE NOT (
        t.src_object_schema <=> s.src_object_schema AND t.src_object <=> s.src_object
        AND t.json_path <=> s.json_path AND t.data_type <=> s.data_type AND t.convert_arg <=> s.convert_arg
        AND t.convert_rule <=> s.convert_rule AND t.is_key <=> s.is_key AND t.key_order <=> s.key_order
        AND t.is_toast <=> s.is_toast AND t.column_order <=> s.column_order
        AND t.is_active <=> s.is_active AND t.description <=> s.description
    )
    """,
)
_apply_seed(
    "place_type", "_v_setup_place_type", PLACE_FQ,
    "t.source_name = s.source_name AND t.place_type = s.place_type", _merge_place_type,
    f"""
    SELECT count(*) AS n
    FROM _v_setup_place_type s
    JOIN {PLACE_FQ} t ON t.source_name = s.source_name AND t.place_type = s.place_type
    WHERE NOT (
        t.business_sector <=> s.business_sector AND t.business_category <=> s.business_category
        AND t.business_category_vi <=> s.business_category_vi AND t.category_group <=> s.category_group
        AND t.range_low_min <=> s.range_low_min AND t.range_high_min <=> s.range_high_min
        AND t.avg_range_min <=> s.avg_range_min AND t.candidate_best_time_code <=> s.candidate_best_time_code
        AND t.canonical_candidate <=> s.canonical_candidate AND t.is_active = true
    )
    """,
)
_apply_seed(
    "amenities", "_v_setup_amenities", AMENITY_FQ,
    "t.source_name = s.source_name AND t.business_sector = s.business_sector "
    "AND t.amenity_group = s.amenity_group AND t.amenity_raw = s.amenity_raw AND t.amenity = s.amenity",
    _merge_amenities,
    f"""
    SELECT count(*) AS n
    FROM _v_setup_amenities s
    JOIN {AMENITY_FQ} t
      ON t.source_name = s.source_name AND t.business_sector = s.business_sector
     AND t.amenity_group = s.amenity_group AND t.amenity_raw = s.amenity_raw AND t.amenity = s.amenity
    WHERE NOT (t.amenity_vi <=> s.amenity_vi AND t.service_flags <=> s.service_flags AND t.is_active = true)
    """,
)


In [ ]:
verify_errors = []
for name, view, table, on_sql, _defined in SEED_PLAN:
    dups = _dup_keys(view, SEED_KEYS[name])
    target_dups = _target_dup_keys(table, SEED_KEYS[name])
    missing = _missing(view, table, on_sql, seed_counts[name]["defined"])
    if dups or target_dups or missing:
        verify_errors.append(f"{name}: dup={dups} dup_target={target_dups} missing={missing}")
    print(
        f"verify seed={name} dup={dups} dup_target={target_dups} "
        f"missing={missing} drift={SEED_RESULTS[name]['drift']}"
    )
reg_pairs = {
    (row["src_schema"], row["src_tbl"], row["trg_schema"], row["trg_tbl"])
    for row in registry_rows
}
cfg_pairs = {
    (src_schema, src_tbl, trg_schema, trg_tbl)
    for pl, src_schema, src_tbl, trg_schema, trg_tbl, *_rest in PIPELINE_CONFIG_ROWS
    if pl == _FL00
}
if reg_pairs != cfg_pairs:
    verify_errors.append(
        f"registry khác pipeline_config extract: thiếu {sorted(reg_pairs - cfg_pairs)[:5]} thừa {sorted(cfg_pairs - reg_pairs)[:5]}"
    )
sources = {(src_schema, src_tbl) for pl, src_schema, src_tbl, *_rest in PIPELINE_CONFIG_ROWS if pl == _FL00}
wm_sources = {(src_schema, src_tbl) for _wid, src_schema, src_tbl, *_rest in WATERMARK_ROWS}
if sources != wm_sources:
    verify_errors.append(f"watermark nguồn extract lệch: config {sorted(sources)} watermark {sorted(wm_sources)}")
if verify_errors:
    raise ConfigError("\n".join(verify_errors))


In [ ]:
summary = {
    "status": "SUCCESS",
    "tables": [spec["fq"] for spec in TABLE_SPECS],
    "seeds": SEED_RESULTS,
}
print(json.dumps(summary))
notebookutils.notebook.exit(json.dumps(summary))
